In [ ]:
# =============================================================================
# CirLog – Circular Logistics Model for Amsterdam Pavement Projects
# Refactored from 02-first-test-model.ipynb
# =============================================================================

In [ ]:
# ------------- Libraries -------------
import os
import json
import math
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point, LineString
import matplotlib.pyplot as plt
import networkx as nx
from pathlib import Path

# ------------- Working directory -------------
os.chdir(r"C:/Users/pkoljensic/OneDrive - Delft University of Technology/Desktop/PYTHON/projects/cirlog")


In [ ]:
# options: , "linear", "baseline", "circular"

SCENARIO = "linear"   #1 no circular strategies 
#SCENARIO = "baseline" #2 current situation with direct/indirect reuse 
#SCENARIO = 'circular' #3 best version of the reuse --- 

In [ ]:
# =================================================================
# 0. SCENARIOS SET-UP
# =================================================================

SCENARIOS = {
    "baseline": {
        "label":               "Baseline",
        "onsite_direct_reuse": 0.05,
        "active_hubs":         ["Opslaglocatie Theemsweg", "Circulaire hub Sluispolderweg"], # here I write which exactly, look at larger hub files; not only current.
        "reuse_rates_path":    "data/raw/reuse_rates/reuse_rates_simple_felix.csv",
    },
    "linear": {
        "label":               "Linear (no reuse)",
        "onsite_direct_reuse": 0.0,
        "active_hubs":         [],          # no hubs active — everything goes to processing
        "reuse_rates_path":    "data/raw/reuse_rates/reuse_rates_linear.csv",   # all zeros
    },
    "circular": {
        "label":               "Circular (high reuse)",
        "onsite_direct_reuse": 0.15,
        "active_hubs":         ["all"],     # all hubs active
        "reuse_rates_path":    "data/raw/reuse_rates/reuse_rates_circular_max.csv", # higher rates
    },
}

# Apply selected scenario — these replace the hardcoded config values
_s = SCENARIOS[SCENARIO]
ONSITE_DIRECT_REUSE = _s["onsite_direct_reuse"]
ACTIVE_HUBS         = _s["active_hubs"]
REUSE_RATES_PATH    = _s["reuse_rates_path"]

In [ ]:
# =============================================================================
# 1. CONFIG
# =============================================================================

CIRCUITY = 1.4          # Straight-line to road distance factor (fallback when no ORS route)
#ONSITE_DIRECT_REUSE = 0.05 # It's in the new datasets for reuse csv, return if needed
BREAKAGE_PCT = 0.10 # add this with downcycling/recycling  # for incoming no breakage rate since anyways max reuse cannot be that high!
MODELED_MATERIALS = {"betontegel", "betonsteen", "gebakken steen"}

# Material density assumptions for translating reuse tonnes to storage volume at hubs.
DENSITY_T_PER_M3 = {
    "betonsteen": 2.3,
    "betontegel": 2.3,
    "gebakken steen": 2.0,
}

# Current hub capacities supplied as area (m2); storage volume assumes a 4 m stacking height.
HUB_CAPACITY_M2 = {
    "Opslaglocatie Theemsweg": 6000,
    "Circulaire hub Sluispolderweg": 2000,
    "Opslaglocatie Buiksloterham": 1000,
}
HUB_STORAGE_HEIGHT_M = 4
HUB_CAPACITY_M3 = {name: area_m2 * HUB_STORAGE_HEIGHT_M for name, area_m2 in HUB_CAPACITY_M2.items()}

# Vehicle classes used in the logistics mix rules.
VEHICLE_NAME_MAP = {
    "Van": "Bestelauto",
    "Medium truck": "Bakwagen",
    "Heavy truck": "Trekker-opleggercombinatie",
}
VEHICLE_AXLES = {"Van": 2, "Medium truck": 3, "Heavy truck": 4}

# Load one vehicle table for both capacity and emissions, including electric values.
VEHICLE_INFO = pd.read_csv("data/raw/logistics/vehicles_info_basic.csv")
VEHICLE_INFO["vehicle_name"] = VEHICLE_INFO["vehicle_name"].astype(str).str.strip()
VEHICLE_INFO["vehicle_type"] = VEHICLE_INFO["vehicle_type"].astype(str).str.strip().str.lower()
VEHICLE_INFO = VEHICLE_INFO[VEHICLE_INFO["transportation_network"] == "road"].copy()

VEHICLE_LIBRARY = {}
for model_name, source_name in VEHICLE_NAME_MAP.items():
    for fuel in ["diesel", "electric"]:
        row = VEHICLE_INFO[(VEHICLE_INFO["vehicle_name"] == source_name) & (VEHICLE_INFO["vehicle_type"] == fuel)]
        if len(row) == 0:
            continue
        data = row.iloc[0]
        VEHICLE_LIBRARY[(model_name, fuel)] = {
            "capacity_t": float(data["capacity_tons"]),
            "co2_tCO2_per_km": float(data["emissions_perKm_tCO2"]),
            "nox_g_per_km": float(data["emissions_perKm_gNOX"]),
            "pm25_g_per_km": float(data["emissions_perKm_gPM2p5"]),
            "pm10_g_per_km": float(data["emissions_perKm_gPM10"]),
        }

# Keep a default capacity per class for quick access in the model.
VEHICLE_CAPACITY = {
    model_name: VEHICLE_LIBRARY[(model_name, "diesel")]["capacity_t"]
    if (model_name, "diesel") in VEHICLE_LIBRARY
    else next(iter(VEHICLE_LIBRARY.values()))["capacity_t"]
    for model_name in VEHICLE_NAME_MAP
}

# Global truck utilisation factors derived from the contractor table pattern.
# They affect trip counts via effective capacity, not the transported tonnes themselves.
INGOING_LOAD_FACTOR = 0.875 # based on contractors data
OUTGOING_LOAD_FACTOR = 0.60 # based on contractors data

USE_ORS_DISTANCES       = True
USE_ORS_EDGES_FOR_USAGE = True

# Reuse ORS outputs for reverse OD legs where we accept symmetric routing.
REVERSED_ORS_KEYS = {
    "projects__hubs": "hubs__projects",
}

# Shared visual language across charts and maps.
FLOW_COLORS = {
    "incoming_new": "#4575b4",
    "incoming_reused": "#f46d43",
    "outgoing_reuse": "#d73027",
    "outgoing_processing": "#878787",
}
FLOW_LABELS = {
    "incoming_new": "Incoming - New",
    "incoming_reused": "Incoming - Reused",
    "outgoing_reuse": "Outgoing - Reuse",
    "outgoing_processing": "Outgoing - Processing",
}
EMISSIONS_COLORS = {
    "total_co2": "#d73027",
    "total_nox": "#4575b4",
    "total_pm25": "#6b7280",
    "total_pm10": "#f59e0b",
}

# --- ORS routing files (one per actor-pair combination) ---
ORS_ROUTES = {
    "suppliers__projects": "data/distances_OD_routes/suppliers_projects_routes.gpkg",
    "hubs__projects":      "data/distances_OD_routes/hubs_projects_routes.gpkg",
    "suppliers__hubs":     "data/distances_OD_routes/suppliers_hubs_routes.gpkg",
    "hubs__processing": "data/distances_OD_routes/hubs_processing_routes.gpkg",
    "projects__processing": "data/distances_OD_routes/projects_processing_routes.gpkg"
}
ORS_EDGES = {
    "suppliers__projects": "data/distances_OD_routes/suppliers_projects_edges.gpkg",
    "hubs__projects": "data/distances_OD_routes/hubs_projects_edges.gpkg",
    "suppliers__hubs": "data/distances_OD_routes/suppliers_hubs_edges.gpkg",
    "hubs__processing": "data/distances_OD_routes/hubs_processing_edges.gpkg",
    "projects__processing": "data/distances_OD_routes/projects_processing_edges.gpkg"
}
ORS_OD_MATRICES = {
    "suppliers__projects": "data/distances_OD_routes/suppliers_projects_od_dist_km.parquet",
    "hubs__projects": "data/distances_OD_routes/hubs_projects_od_dist_km.parquet",
    "suppliers__hubs":     "data/distances_OD_routes/suppliers_hubs_od_dist_km.parquet",
    "hubs__processing": "data/distances_OD_routes/hubs_processing_od_dist_km.parquet",
    "projects__processing": "data/distances_OD_routes/projects_processing_od_dist_km.parquet"
}

os.makedirs(f"test_model/{SCENARIO}", exist_ok=True)
EDGE_USAGE_CSV  = f"test_model/{SCENARIO}/logistics_edge_usage.csv"
EDGE_USAGE_GPKG = f"test_model/{SCENARIO}/logistics_edge_usage.gpkg"


EDGE_CACHE_DIR = Path("data/distances_OD_routes/cache")
EDGE_INDEX_GPKG = EDGE_CACHE_DIR / "edges_index.gpkg"
ROAD_MATRIX_NPY = EDGE_CACHE_DIR / "road_matrix.npy"
EDGE_CACHE_META = EDGE_CACHE_DIR / "edge_cache_meta.json"
EDGE_CACHE_NOTEBOOK = "notebooks/05-precompute-edge-cache.ipynb"


In [ ]:
# =============================================================================
# 2. HELPERS
# =============================================================================

def norm_material(s: str) -> str:
    """Normalise raw material strings to canonical names.
    Handles plain strings and Python list-strings like "['Gebakken steen']".
    """
    if not isinstance(s, str):
        return s
    t = s.strip().strip("[]'\" ").lower()
    if "gebakken" in t or "brick" in t:
        return "gebakken steen"
    if "betontegel" in t or "tegel" in t:
        return "betontegel"
    if "betonsteen" in t or "paver" in t:
        return "betonsteen"
    return t

def ensure_crs(gdf, to_crs="EPSG:28992") -> gpd.GeoDataFrame:
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(to_crs)

def pointize(gdf) -> gpd.GeoDataFrame:
    """Replace polygon/line geometries with their centroid."""
    if len(gdf) == 0:
        return gdf
    if gdf.geom_type.iloc[0].lower() in ("polygon", "multipolygon", "linestring", "multilinestring"):
        gdf = gdf.copy()
        gdf["geometry"] = gdf.geometry.centroid
    return gdf

def distance_km(a: Point, b: Point) -> float:
    return (a.distance(b) / 1000.0) * CIRCUITY

def effective_capacity(capacity_t: float, load_factor: float) -> float:
    """Convert nominal vehicle capacity into usable payload capacity."""
    return max(capacity_t * load_factor, 1e-9)


def trips_required(tonnes: float, capacity_t: float, load_factor: float = 1.0) -> int:
    """Trip count based on usable vehicle capacity after applying a load factor."""
    return math.ceil(tonnes / effective_capacity(capacity_t, load_factor)) if tonnes > 0 else 0

def _id_str(x):
    return None if pd.isna(x) else str(x).strip()

def od_lookup_keys(od_key: str) -> list[str]:
    keys = [od_key]
    base_key = REVERSED_ORS_KEYS.get(od_key)
    if base_key and base_key not in keys:
        keys.append(base_key)
    return keys

def reverse_ors_aliases(source_key: str) -> list[str]:
    return [alias for alias, base in REVERSED_ORS_KEYS.items() if base == source_key]

def load_ors_od_matrices(paths: dict) -> dict:
    result = {}
    for od_key, path in paths.items():
        if not USE_ORS_DISTANCES or not os.path.exists(path):
            print(f"  OD matrix not found or disabled: {path}")
            result[od_key] = {}
            continue
        try:
            df = pd.read_csv(path, index_col=0)
            df.index = df.index.map(str).str.strip()
            df.columns = df.columns.map(str).str.strip()
            lkp = {
                (orig, dest): float(df.loc[orig, dest])
                for orig in df.index
                for dest in df.columns
                if not pd.isna(df.loc[orig, dest])
            }
            result[od_key] = lkp
            print(f"  Loaded {len(lkp)} OD pairs ({len(df)} origins × {len(df.columns)} destinations) from {path}")
        except Exception as e:
            print(f"  Could not load {path}: {e}")
            result[od_key] = {}
    return result

def load_ors_routes(paths: dict) -> dict:
    result = {}
    for od_key, path in paths.items():
        if not os.path.exists(path):
            print(f"  Route file not found: {path}")
            result[od_key] = gpd.GeoDataFrame()
            continue
        gdf = gpd.read_file(path)
        result[od_key] = gdf
        print(f"  Loaded {len(gdf)} routes from {path}")
    return result

def load_ors_edges(paths: dict) -> dict:
    result = {}
    for od_key, path in paths.items():
        if not os.path.exists(path):
            print(f"  Edge file not found: {path}")
            result[od_key] = {}
            continue
        gdf = gpd.read_file(path)
        lkp = {}
        for _, r in gdf.iterrows():
            k = (str(r["origin_id"]).strip(), str(r["destination_id"]).strip())
            ek = _edge_key(r.geometry)
            if ek:
                lkp.setdefault(k, []).append(ek)
        result[od_key] = lkp
        print(f"  Loaded edges for {len(lkp)} OD pairs from {path}")
    return result

def _edge_key(geom):
    """Canonical (undirected) edge key from a LineString geometry."""
    if geom is None or geom.is_empty:
        return None
    try:
        c = list(geom.coords)
    except Exception:
        return None
    if len(c) < 2:
        return None
    a = (round(float(c[0][0]), 6), round(float(c[0][1]), 6))
    b = (round(float(c[-1][0]), 6), round(float(c[-1][1]), 6))
    return (a, b) if a <= b else (b, a)

def normalize_text(value) -> str:
    if value is None or pd.isna(value):
        return ""
    text = str(value).strip().lower()
    for old, new in {
        "&": " and ",
        "-": " ",
        "_": " ",
        "(": " ",
        ")": " ",
        "/": " ",
        ".": " ",
        ",": " ",
        "'": "",
    }.items():
        text = text.replace(old, new)
    return " ".join(text.split())

CONTRACTOR_ALIASES = {
    "beentjes": "beentjes gww",
    "beentjes gww": "beentjes gww",
    "compeer": "compeer",
    "griekspoor": "griekspoor",
    "h van wijk": "h van wijk",
    "koot": "koot infrawerken",
    "koot infrawerken": "koot infrawerken",
    "markus veekens": "markus verkeens",
    "markus verkeens": "markus verkeens",
    "reimert": "reimert",
    "rutger": "rutte groep",
    "rutte groep": "rutte groep",
    "schot": "schot infra",
    "schot infra": "schot infra",
    "van wijk nieuwegein": "van wijk nieuwegein",
}

LOCATION_COLUMN_PAIRS = [
    ("loc_reuse_ams1", "rate_reuse_ams1"),
    ("loc_reuse_ams2", "rate_reuse_ams2"),
    ("loc_reuse_outside", "rate_reuse_outside"),
    ("loc_recycling", "rate_recycling"),
    ("loc_down1", "rate_down1"),
    ("loc_down2", "rate_down2"),
    ("loc_down3", "rate_down3"),
]

def canonical_contractor_name(value) -> str:
    key = normalize_text(value)
    return CONTRACTOR_ALIASES.get(key, key)

def load_complex_reuse_tables(base_dir: str = "data/raw/reuse_rates") -> dict[str, pd.DataFrame]:
    files = {
        "betontegel": "data_reuse_complex_betontegel.csv",
        "betonsteen": "data_reuse_complex_betonsteen.csv",
        "gebakken steen": "data_reuse_complex_gebakkensteen.csv",
    }
    tables = {}
    for material, filename in files.items():
        df = pd.read_csv(Path(base_dir) / filename)
        if "Unnamed: 0" in df.columns:
            df = df.drop(columns="Unnamed: 0")
        for _, rate_col in LOCATION_COLUMN_PAIRS:
            df[rate_col] = pd.to_numeric(df[rate_col], errors="coerce").fillna(0.0)
        df["contractor_norm"] = df["contractor"].apply(canonical_contractor_name)
        tables[material] = df
    return tables


def edge_cache_source_snapshot(paths: dict) -> dict:
    snapshot = {}
    for od_key, path in paths.items():
        p = Path(path)
        info = {
            "path": p.as_posix(),
            "exists": p.exists(),
        }
        if p.exists():
            stat = p.stat()
            info["mtime_ns"] = int(stat.st_mtime_ns)
            info["size"] = int(stat.st_size)
        snapshot[od_key] = info
    return snapshot


def load_edge_usage_cache():
    required = [EDGE_INDEX_GPKG, ROAD_MATRIX_NPY, EDGE_CACHE_META]
    missing = [str(path) for path in required if not Path(path).exists()]
    if missing:
        missing_text = ", ".join(missing)
        raise RuntimeError(
            f"Edge cache missing ({missing_text}). Run {EDGE_CACHE_NOTEBOOK} first to build edges_gdf and road_matrix."
        )

    edges_gdf = gpd.read_file(EDGE_INDEX_GPKG)
    if "edge_id" in edges_gdf.columns:
        edges_gdf = edges_gdf.set_index("edge_id")

    road_matrix = np.load(ROAD_MATRIX_NPY, allow_pickle=True)
    meta = json.loads(Path(EDGE_CACHE_META).read_text(encoding="utf-8"))
    current_snapshot = edge_cache_source_snapshot(ORS_EDGES)

    changed_sources = []
    cached_snapshot = meta.get("source_files", {})
    for od_key, info in current_snapshot.items():
        if cached_snapshot.get(od_key) != info:
            changed_sources.append(od_key)

    alias_changed = meta.get("reversed_ors_keys", {}) != REVERSED_ORS_KEYS
    key_changed = meta.get("edge_key_version") != "v1"

    print(f"Loaded edge cache: {len(edges_gdf)} unique edges, {len(road_matrix)} OD routes")
    if changed_sources or alias_changed or key_changed:
        print("Rebuild edge cache recommended.")
        if changed_sources:
            print("  Changed edge sources:", changed_sources)
        if alias_changed:
            print("  Reverse OD alias settings changed.")
        if key_changed:
            print("  Edge key version changed.")
    else:
        print("Edge cache is up to date.")

    print("No rebuild needed for scenario, reuse-rate, contractor, vehicle, or emissions changes.")
    print("No rebuild needed if a flow switches to another hub or processing site that is already present in the cached ORS edge files.")
    print("Rebuild only if you regenerate any ORS *edges.gpkg files, add a new OD edge file, or change the edge-key / reverse-alias logic.")
    return edges_gdf, road_matrix, meta


In [ ]:
# =============================================================================
# 3. LOAD DATA
# =============================================================================

def load_geo(path, **kwargs):
    """Load a GeoPackage, reprojects to RD New and pointizes."""
    return ensure_crs(pointize(gpd.read_file(path, **kwargs)))

# Spatial layers
projects    = load_geo("data/cleaned/projects_with_contractor.gpkg")
suppliers   = load_geo("data/cleaned/suppliers_gdf.gpkg")
hubs        = load_geo("data/processed/hubs_storage_current.gpkg")
processing  = load_geo("data/cleaned/processing_locations_gdf_clean.gpkg")
processing  = processing.dropna(subset=["material_type"]).copy()

# Tabular
reuse = (pd.read_csv(REUSE_RATES_PATH)
           .rename(columns={"Pavement material": "material",
                             "Incoming reuse (%)": "incoming_reuse_pct",
                             "Outgoing reuse (%)": "outgoing_reuse_pct"})
           .assign(material_norm=lambda d: d["material"].apply(norm_material))
           [["material_norm", "incoming_reuse_pct", "outgoing_reuse_pct"]])

COMPLEX_REUSE_TABLES = load_complex_reuse_tables()

# Normalise IDs
projects["year_bucket"]    = projects["Start_jaar"].fillna("Unknown")
projects["tonnage"]        = pd.to_numeric(projects["tonnage"], errors="coerce").fillna(0)
suppliers["material_norm"] = suppliers["material_type"].apply(norm_material)

print(f"Loaded projects: {len(projects)}")
print(f"Loaded suppliers: {len(suppliers)}")
print(f"Loaded hubs: {len(hubs)}")
print(f"Loaded processing locations: {len(processing)}")
print(f"Loaded complex processing tables for: {sorted(COMPLEX_REUSE_TABLES)}")

# ORS routing data — one lookup per actor-pair type
print("Loading ORS routing data...")
ORS_OD = load_ors_od_matrices(ORS_OD_MATRICES)   # distance lookups
ORS_ROUTES_GDF = load_ors_routes(ORS_ROUTES)     # full route lines (for visualisation)
# Edge files are loaded and indexed in section 10 using the BIMZEC integer-ID pattern

REQUIRED_ORS_KEYS = {
    "suppliers__projects",
    "hubs__projects",
    "projects__processing",
}
if len(ACTIVE_HUBS) > 0:
    REQUIRED_ORS_KEYS.add("projects__hubs")

missing_od = sorted(
    k for k in REQUIRED_ORS_KEYS
    if not any(ORS_OD.get(candidate) for candidate in od_lookup_keys(k))
)
missing_routes = sorted(
    k for k in REQUIRED_ORS_KEYS
    if not any(len(ORS_ROUTES_GDF.get(candidate, gpd.GeoDataFrame())) > 0 for candidate in od_lookup_keys(k))
)

if missing_od:
    print(f"Warning: missing ORS OD matrix lookups for: {missing_od}")
if missing_routes:
    print(f"Warning: missing ORS route geometries for: {missing_routes}")
if "projects__hubs" in missing_od or "projects__hubs" in missing_routes:
    print("Note: project -> hub flows will fall back to circuity unless you generate and register projects__hubs ORS outputs.")

def leg_distance_km(od_key: str, orig_id: str, dest_id: str, orig_pt: Point, dest_pt: Point):
    """
    Return (distance_km, source_label).
    Priority: OD CSV matrix (values in km) -> ORS route gpkg -> Euclidean x CIRCUITY fallback.
    """
    o, d = _id_str(orig_id), _id_str(dest_id)
    # 1. Try OD distance matrix (CSV pivot, values already in km)
    for candidate_key in od_lookup_keys(od_key):
        lkp = ORS_OD.get(candidate_key, {})
        km = lkp.get((o, d)) or lkp.get((d, o))
        if km is not None:
            return km, "ors_matrix"
    # 2. Try route gpkg (distance_m attribute -> convert to km)
    for candidate_key in od_lookup_keys(od_key):
        routes = ORS_ROUTES_GDF.get(candidate_key, gpd.GeoDataFrame())
        if len(routes) == 0:
            continue
        match = routes[(routes["origin_id"] == o) & (routes["destination_id"] == d)]
        if len(match) == 0:
            match = routes[(routes["origin_id"] == d) & (routes["destination_id"] == o)]
        if len(match) > 0:
            return float(match.iloc[0]["distance_m"]) / 1000.0, "ors_route"
    # 3. Euclidean fallback
    return distance_km(orig_pt, dest_pt), "circuity"


### Notebook cleanup note
- This copy uses `processing_locations_gdf_clean.gpkg` consistently.
- Scratch inspection cells were removed to keep the model flow linear.
- ORS loading now prints explicit warnings when a required OD pair is missing.


In [ ]:
# =============================================================================
# 4. BUILD PROJECT × MATERIAL TABLE
# =============================================================================

def split_materials(row) -> list[tuple[str, float]]:
    """Return [(material, tonnes_share), ...] for each modelled material in a project row."""
    mats = [norm_material(x) for x in str(row["TYPE_EXTRA"]).split(",")]
    mats = [m for m in mats if m in MODELED_MATERIALS]
    if not mats:
        return []
    share = row["tonnage"] / len(mats) if row["tonnage"] else 0
    return [(m, share) for m in mats]

proj_rows = [
    {"project_key": idx, "Projnaam": r["Projnaam"],
     "Aannemer": r.get("Aannemer"),
     "Stadsdeel": r.get("Stadsdeel"),
     "material_norm": m, "tonnes": float(t),
     "geometry": r["geometry"], "year": r["year_bucket"]}
    for idx, r in projects.iterrows()
    for m, t in split_materials(r)
]
proj_df = gpd.GeoDataFrame(proj_rows, geometry="geometry", crs=projects.crs)

# Attach reuse rates
proj_df = proj_df.merge(reuse, on="material_norm", how="left")
proj_df["incoming_reuse_pct"] = proj_df["incoming_reuse_pct"].fillna(0)
proj_df["outgoing_reuse_pct"] = proj_df["outgoing_reuse_pct"].fillna(0)



In [ ]:
# Optional cleanup of incomplete future years
# Set EXCLUDED_YEARS to [] to keep all projects.
EXCLUDED_YEARS = [2032]

if EXCLUDED_YEARS:
    proj_df = proj_df[~proj_df["year"].isin(EXCLUDED_YEARS)].copy()
    print(f"Excluded years from modelling: {EXCLUDED_YEARS}")
else:
    print("Excluded years from modelling: none")



In [ ]:
# =============================================================================
# 5. TONNAGE PARTITIONING
# =============================================================================

proj_df["tonnes_onsite_reuse"]     = proj_df["tonnes"] * ONSITE_DIRECT_REUSE
proj_df["tonnes_for_inbound"]      = proj_df["tonnes"] - proj_df["tonnes_onsite_reuse"]
proj_df["tonnes_reused_in"]        = proj_df["tonnes_for_inbound"] * (proj_df["incoming_reuse_pct"] / 100)
proj_df["tonnes_new_in"]           = proj_df["tonnes_for_inbound"] - proj_df["tonnes_reused_in"]
proj_df["tonnes_breakage"]         = proj_df["tonnes"] * BREAKAGE_PCT
proj_df["tonnes_out_for_reuse"]    = proj_df["tonnes"] * (proj_df["outgoing_reuse_pct"] / 100)
proj_df["tonnes_out_for_processing"] = np.maximum(
    proj_df["tonnes"] - proj_df["tonnes_out_for_reuse"], proj_df["tonnes_breakage"]
)

In [ ]:
# =============================================================================
# 6. HUB LOOKUP
# =============================================================================

def find_hub(df, keys):
    mask = pd.Series(False, index=df.index)
    for c in df.select_dtypes("object").columns:
        mask |= df[c].str.lower().str.contains("|".join(keys), na=False)
    return df[mask].copy()

beentjes = find_hub(hubs, ["beentjes", "sluispolderweg"]).head(1)
stads    = find_hub(hubs, ["stadswerken", "stadswerk"]).head(1)
if beentjes.empty and len(hubs) > 0: beentjes = hubs.head(1)
if stads.empty    and len(hubs) > 1: stads    = hubs.iloc[[1]]

def _hub_info(df, fallback_name):
    pt   = df.geometry.iloc[0] if not df.empty else None
    name = df.get("name", pd.Series([fallback_name])).iloc[0] if not df.empty else fallback_name
    return pt, _id_str(name)

beentjes_pt, beentjes_name = _hub_info(beentjes, "Beentjes")
stads_pt,    stads_name    = _hub_info(stads,    "Stadswerken hub")

STONE_MATS = {"betonsteen", "gebakken steen"}   # materials routed via Beentjes hub

# Updates

def hub_for(mat):
    pt, name = (beentjes_pt, beentjes_name) if mat in STONE_MATS else (stads_pt, stads_name)
    if ACTIVE_HUBS != ["all"] and name not in ACTIVE_HUBS:
        return None, None
    return pt, name

In [ ]:
# =============================================================================
# 7. SUPPLIER + PROCESSING + VEHICLE MIX RULES
# =============================================================================

supplier_rng = np.random.default_rng(2026)
processing_rng = np.random.default_rng(2027)

PROCESSING_COLUMN_RULES = [
    ("loc_recycling", "rate_recycling", "recycling"),
    ("loc_down1", "rate_down1", "down1"),
    ("loc_down2", "rate_down2", "down2"),
    ("loc_down3", "rate_down3", "down3"),
]

def supplier_candidates(mat: str):
    cand = suppliers[suppliers["material_norm"] == mat].copy()
    if "company" in cand.columns:
        cand["company_norm"] = cand["company"].astype(str).str.strip().str.lower()
    else:
        cand["company_norm"] = ""
    return cand

# 09 update: supplier selection is company-based, not nearest-based.
def select_supplier(mat: str):
    cand = supplier_candidates(mat)
    if len(cand) == 0:
        return None, None
    if mat == "gebakken steen":
        chosen_company = supplier_rng.choice(["vandersanden", "wienerberger"], p=[0.7, 0.3])
        company_cand = cand[cand["company_norm"] == chosen_company]
        if len(company_cand) > 0:
            cand = company_cand
    elif mat in {"betontegel", "betonsteen"}:
        company_cand = cand[cand["company_norm"] == "struyk verwo"]
        if len(company_cand) > 0:
            cand = company_cand
    idx = int(supplier_rng.choice(cand.index.to_numpy()))
    return cand.geometry.loc[idx], cand.loc[idx]

proc_name_col = next((c for c in processing.columns if "name" in c.lower()), processing.columns[0])
processing_lookup = {
    normalize_text(row[proc_name_col]): row
    for _, row in processing.iterrows()
}

def choose_processing_destination(material: str, contractor: str):
    table = COMPLEX_REUSE_TABLES.get(material)
    if table is None:
        return None
    contractor_norm = canonical_contractor_name(contractor)
    matches = table[table["contractor_norm"] == contractor_norm]
    if len(matches) == 0:
        return None
    row = matches.iloc[0]
    options = []
    for loc_col, rate_col, channel in PROCESSING_COLUMN_RULES:
        location_name = row.get(loc_col)
        rate = pd.to_numeric(row.get(rate_col), errors="coerce")
        if pd.isna(location_name) or pd.isna(rate) or float(rate) <= 0:
            continue
        proc_row = processing_lookup.get(normalize_text(location_name))
        if proc_row is None:
            continue
        options.append({
            "channel": channel,
            "name": _id_str(proc_row[proc_name_col]),
            "geometry": proc_row.geometry,
        })
    if not options:
        return None
    return options[int(processing_rng.integers(0, len(options)))]

def is_centrum(stadsdeel: str) -> bool:
    return normalize_text(stadsdeel) == "centrum"

# 09 update: logistics mix is applied per flow row and represents the mix of
# deliveries for that flow, not a direct tonne split across vehicles.
def logistics_mix(flow_type: str, stadsdeel: str):
    centrum = is_centrum(stadsdeel)
    if flow_type == "incoming_new":
        if centrum:
            return [
                {"vehicle": "Heavy truck", "fuel": "electric", "share": 0.40},
                {"vehicle": "Heavy truck", "fuel": "diesel", "share": 0.10},
                {"vehicle": "Medium truck", "fuel": "electric", "share": 0.40},
                {"vehicle": "Medium truck", "fuel": "diesel", "share": 0.10},
            ]
        return [
            {"vehicle": "Heavy truck", "fuel": "diesel", "share": 0.90},
            {"vehicle": "Heavy truck", "fuel": "electric", "share": 0.10},
        ]
    if centrum:
        return [
            {"vehicle": "Van", "fuel": "electric", "share": 0.48},
            {"vehicle": "Van", "fuel": "diesel", "share": 0.12},
            {"vehicle": "Medium truck", "fuel": "electric", "share": 0.32},
            {"vehicle": "Medium truck", "fuel": "diesel", "share": 0.08},
        ]
    return [
        {"vehicle": "Medium truck", "fuel": "electric", "share": 0.35},
        {"vehicle": "Medium truck", "fuel": "diesel", "share": 0.35},
        {"vehicle": "Van", "fuel": "electric", "share": 0.15},
        {"vehicle": "Van", "fuel": "diesel", "share": 0.15},
    ]

def allocate_trip_mix(total_tonnes: float, mix: list[dict], load_factor: float):
    """Allocate integer trips across vehicle/fuel parts using delivery shares.

    The mix shares describe how often a vehicle/fuel combination is used among
    deliveries, not how much tonnage it takes. We therefore compute one total
    trip count from the weighted effective capacity, then distribute trips by
    the delivery shares and split tonnes proportionally to available capacity.
    """
    if total_tonnes <= 0:
        return []

    prepared = []
    for part in mix:
        vehicle_data = VEHICLE_LIBRARY.get((part["vehicle"], part["fuel"]))
        if vehicle_data is None:
            continue
        effective_capacity_t = effective_capacity(vehicle_data["capacity_t"], load_factor)
        prepared.append({
            **part,
            "capacity_t": vehicle_data["capacity_t"],
            "effective_capacity_t": effective_capacity_t,
        })

    if not prepared:
        return []

    weighted_effective_capacity = sum(p["share"] * p["effective_capacity_t"] for p in prepared)
    if weighted_effective_capacity <= 0:
        return []

    total_trips = max(1, math.ceil(total_tonnes / weighted_effective_capacity))
    expected_trips = [p["share"] * total_trips for p in prepared]
    allocated_trips = [math.floor(x) for x in expected_trips]
    remaining = total_trips - sum(allocated_trips)
    order = sorted(
        range(len(prepared)),
        key=lambda i: (expected_trips[i] - allocated_trips[i], prepared[i]["share"]),
        reverse=True,
    )
    while remaining > 0 and order:
        for idx in order:
            if remaining == 0:
                break
            allocated_trips[idx] += 1
            remaining -= 1

    total_available_capacity = 0.0
    for idx, trips in enumerate(allocated_trips):
        prepared[idx]["trips"] = int(trips)
        total_available_capacity += prepared[idx]["trips"] * prepared[idx]["effective_capacity_t"]

    if total_available_capacity <= 0:
        return []

    allocated = []
    for part in prepared:
        if part["trips"] <= 0:
            continue
        tonnes_share = total_tonnes * ((part["trips"] * part["effective_capacity_t"]) / total_available_capacity)
        allocated.append({
            **part,
            "tonnes": tonnes_share,
        })
    return allocated


In [ ]:
# =============================================================================
# 8. BUILD FLOW LEGS
# =============================================================================

# Main update in 09:
# - supplier choice follows company rules by material
# - trip counts now use one global load factor for inbound flows and one for outbound flows
# - outgoing processing is randomized per project-material flow using contractor/material processing lists
# - each flow row is split into vehicle/fuel subflows using the logistics mix assumptions
# - capacities now come directly from vehicles_info_basic.csv per vehicle + fuel
# - vehicle shares are treated as delivery shares, so totals stay internally consistent

def build_flow_records(r, flow_type, od_key, orig_type, orig_name, orig_pt, dest_type, dest_name, dest_pt, tonnes, channel=None):
    """Expand one logical flow into vehicle/fuel subflows."""
    d, dsrc = leg_distance_km(od_key, orig_name, dest_name, orig_pt, dest_pt)
    load_factor = INGOING_LOAD_FACTOR if flow_type in {"incoming_new", "incoming_reused"} else OUTGOING_LOAD_FACTOR
    mix = logistics_mix(flow_type, r.get("Stadsdeel"))
    allocated_parts = allocate_trip_mix(tonnes, mix, load_factor)

    subrecords = []
    for part in allocated_parts:
        tonnes_share = part["tonnes"]
        trips = part["trips"]
        capacity_t = part["capacity_t"]
        subrecords.append({
            "year": r["year"],
            "project_key": r["project_key"],
            "project_name": _id_str(r.get("Projnaam")),
            "contractor": _id_str(r.get("Aannemer")),
            "stadsdeel": _id_str(r.get("Stadsdeel")),
            "material": r["material_norm"],
            "flow": flow_type,
            "od_key": od_key,
            "origin_id": orig_name,
            "destination_id": dest_name,
            "origin": orig_type,
            "origin_name": orig_name,
            "destination": dest_type,
            "channel": channel,
            "fuel": part["fuel"],
            "vehicle": part["vehicle"],
            "vehicle_class": part["vehicle"],
            "vehicle_source_name": VEHICLE_NAME_MAP[part["vehicle"]],
            "mix_share": part["share"],
            "capacity_t": capacity_t,
            "load_factor": load_factor,
            "tonnes": tonnes_share,
            "trips": trips,
            "distance_source": dsrc,
            "km": d * 2 * trips,
            "tkm": d * 2 * tonnes_share,
        })
    return subrecords

records = []
for _, r in proj_df.iterrows():
    proj_pt = r["geometry"]
    proj_name = _id_str(r.get("Projnaam"))
    mat = r["material_norm"]
    contractor = r.get("Aannemer")

    hub_pt, hub_name = hub_for(mat)

    if r["tonnes_reused_in"] > 0 and hub_pt is not None:
        records.extend(build_flow_records(r, "incoming_reused", "hubs__projects",
                                          "hub", hub_name, hub_pt,
                                          "project", proj_name, proj_pt,
                                          r["tonnes_reused_in"]))

    if r["tonnes_new_in"] > 0:
        sgeom, srow = select_supplier(mat)
        if sgeom is not None:
            sname = _id_str(srow.get("name", "supplier"))
            records.extend(build_flow_records(r, "incoming_new", "suppliers__projects",
                                              "supplier", sname, sgeom,
                                              "project", proj_name, proj_pt,
                                              r["tonnes_new_in"]))

    if r["tonnes_out_for_reuse"] > 0 and hub_pt is not None:
        records.extend(build_flow_records(r, "outgoing_reuse", "projects__hubs",
                                          "project", proj_name, proj_pt,
                                          "hub", hub_name, hub_pt,
                                          r["tonnes_out_for_reuse"]))

    if r["tonnes_out_for_processing"] > 0 and len(processing) > 0:
        processing_choice = choose_processing_destination(mat, contractor)
        if processing_choice is None:
            fallback_idx = int(processing_rng.integers(0, len(processing)))
            proc_row = processing.iloc[fallback_idx]
            proc_pt = proc_row.geometry
            proc_name = _id_str(proc_row[proc_name_col])
            channel = "fallback_random"
        else:
            proc_pt = processing_choice["geometry"]
            proc_name = processing_choice["name"]
            channel = processing_choice["channel"]

        records.extend(build_flow_records(r, "outgoing_processing", "projects__processing",
                                          "project", proj_name, proj_pt,
                                          "processing", proc_name, proc_pt,
                                          r["tonnes_out_for_processing"], channel=channel))

flows_df = pd.DataFrame(records)

In [ ]:
# =============================================================================
# 9. HUB STORAGE REQUIREMENTS
# =============================================================================

# Translate hub-related reuse flows from tonnes to required storage volume (m3)
# and compare them with the current hub capacities (m2 and m3).

hub_flows = flows_df[flows_df["flow"].isin(["incoming_reused", "outgoing_reuse"])].copy()

if len(hub_flows) == 0:
    hub_storage_summary = pd.DataFrame()
    print("No hub reuse flows available for storage calculation.")
else:
    hub_flows["hub_name"] = np.where(
        hub_flows["flow"] == "incoming_reused",
        hub_flows["origin_id"],
        hub_flows["destination_id"],
    )
    hub_flows["density_t_per_m3"] = hub_flows["material"].map(DENSITY_T_PER_M3)
    hub_flows["hub_volume_m3"] = np.where(
        hub_flows["density_t_per_m3"].gt(0),
        hub_flows["tonnes"] / hub_flows["density_t_per_m3"],
        np.nan,
    )

    hub_storage_summary = (
        hub_flows.groupby(["year", "hub_name", "material", "flow"], as_index=False)
        .agg(tonnes=("tonnes", "sum"),
             hub_volume_m3_required=("hub_volume_m3", "sum"))
    )

    hub_capacity_df = pd.DataFrame([
        {
            "hub_name": name,
            "current_capacity_m2": HUB_CAPACITY_M2.get(name),
            "current_capacity_m3": HUB_CAPACITY_M3.get(name),
        }
        for name in sorted(set(hub_storage_summary["hub_name"]).union(HUB_CAPACITY_M2))
    ])

    hub_storage_summary = hub_storage_summary.merge(hub_capacity_df, on="hub_name", how="left")
    hub_storage_totals = (
        hub_storage_summary.groupby(["year", "hub_name"], as_index=False)
        .agg(total_hub_volume_m3_required=("hub_volume_m3_required", "sum"),
             current_capacity_m2=("current_capacity_m2", "first"),
             current_capacity_m3=("current_capacity_m3", "first"))
    )
    hub_storage_totals["capacity_gap_m3"] = (
        hub_storage_totals["current_capacity_m3"] - hub_storage_totals["total_hub_volume_m3_required"]
    )

    print("Hub storage summary (m3 required vs current capacity):")
    print(hub_storage_totals.to_string(index=False))

    hub_storage_summary.to_csv(f"test_model/{SCENARIO}/hub_storage_by_material.csv", index=False)
    hub_storage_totals.to_csv(f"test_model/{SCENARIO}/hub_storage_totals.csv", index=False)



In [ ]:
# =============================================================================
# 10. EDGE USAGE  (cached BIMZEC-style integer edge ID pattern)
# =============================================================================
# Static preprocessing now lives in notebooks/05-precompute-edge-cache.ipynb:
#   1. Load all ORS edge gpkg files
#   2. Build one stable edges_gdf with integer edge_id
#   3. Build one cached road_matrix for OD route -> edge_id arrays
#
# This cell now only does the dynamic scenario step:
#   - read flows_df
#   - add trips onto cached edge_ids
#   - export roads_used_active / edge_usage_gdf
#
# Rebuild the cache only when the static route-edge network changes.
# No rebuild is needed for scenario changes, supplier/contractor choices,
# reuse rates, vehicle mix, or a different hub selection if that OD route is
# already represented in the cached ORS edge files.
#
# Performance note:
# - monthly modelling increases flows_df rows a lot
# - so we aggregate flows first and use a dict lookup for cached routes
# - this avoids scanning road_matrix and the full edge index for every row

VEHICLE_WEIGHT = {"Van": 1, "Medium truck": 2, "Heavy truck": 3}

if USE_ORS_EDGES_FOR_USAGE and len(flows_df) > 0:
    edges_gdf, road_matrix, edge_cache_meta = load_edge_usage_cache()

    if len(road_matrix) > 0:
        route_lookup = {
            (row[0], row[1], row[2]): np.asarray(row[3], dtype=np.int32)
            for row in road_matrix
        }

        flow_edges = flows_df[["od_key", "origin_id", "destination_id", "vehicle", "trips"]].copy()
        flow_edges["origin_id"] = flow_edges["origin_id"].map(_id_str)
        flow_edges["destination_id"] = flow_edges["destination_id"].map(_id_str)
        flow_edges["trips"] = pd.to_numeric(flow_edges["trips"], errors="coerce").fillna(0).astype(int)
        flow_edges = flow_edges[flow_edges["trips"] > 0].copy()
        flow_edges["vehicle_weight"] = flow_edges["vehicle"].map(VEHICLE_WEIGHT).fillna(3).astype(float)
        flow_edges["weighted_trip_value"] = flow_edges["trips"] * flow_edges["vehicle_weight"]

        grouped_flows = (
            flow_edges.groupby(["od_key", "origin_id", "destination_id"], as_index=False)
            .agg(
                trips=("trips", "sum"),
                weighted_trips=("weighted_trip_value", "sum"),
            )
        )

        ntrip_array = np.zeros(len(edges_gdf), dtype=np.int32)
        weighted_array = np.zeros(len(edges_gdf), dtype=np.float32)
        missing_routes = 0

        for _, r in grouped_flows.iterrows():
            key = (r["od_key"], r["origin_id"], r["destination_id"])
            edge_ids = route_lookup.get(key)
            if edge_ids is None or len(edge_ids) == 0:
                missing_routes += 1
                continue

            np.add.at(ntrip_array, edge_ids, int(r["trips"]))
            np.add.at(weighted_array, edge_ids, float(r["weighted_trips"]))

        roads_used = pd.DataFrame({
            "edge_id": np.arange(len(edges_gdf)),
            "nTrips": ntrip_array,
            "weighted_trips": weighted_array,
        }).set_index("edge_id")

        roads_used_active = roads_used[roads_used["nTrips"] > 0].copy()
        roads_used_active = roads_used_active.sort_values(["nTrips", "weighted_trips"], ascending=False)
        roads_used_active.reset_index().to_csv(EDGE_USAGE_CSV, index=False)

        edge_usage_gdf = edges_gdf.loc[roads_used_active.index].join(roads_used_active)
        edge_usage_gdf = edge_usage_gdf[edge_usage_gdf.geometry.notna()]
        if len(edge_usage_gdf):
            edge_usage_gdf.to_file(EDGE_USAGE_GPKG, driver="GPKG")

        print(f"Grouped {len(flow_edges)} flow rows into {len(grouped_flows)} OD route lookups.")
        if missing_routes:
            print(f"Missing cached route match for {missing_routes} grouped OD routes.")
        print(f"Edge usage saved: {len(roads_used_active)} active edges (out of {len(edges_gdf)} total).")
    else:
        edge_usage_gdf = gpd.GeoDataFrame()
        print("Road matrix is empty — edge usage skipped.")
else:
    edge_usage_gdf = gpd.GeoDataFrame()
    print("No edge usage computed (edge usage disabled or no flows_df rows).")



In [ ]:
# =============================================================================
# 11. KPI CHARTS
# =============================================================================

# Recompute totals from the current flows_df so the total line always matches
# the per-flow breakdown after the latest model changes.
os.makedirs(f"test_model/{SCENARIO}", exist_ok=True)

flows_df["year"] = flows_df["year"].astype(str)
for col in ["km", "tkm", "trips", "tonnes"]:
    flows_df[col] = pd.to_numeric(flows_df[col], errors="coerce").fillna(0)

kpis = (flows_df.groupby("year", as_index=False)
        .agg(total_km=("km", "sum"),
             total_tkm=("tkm", "sum"),
             total_trips=("trips", "sum"),
             total_tonnes=("tonnes", "sum"))
        .sort_values("year"))

flows_df.to_csv(f"test_model/{SCENARIO}/logistics_flows_per_project_year.csv", index=False)
kpis.to_csv(f"test_model/{SCENARIO}/logistics_kpis_per_year.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
kpi_plots = [
    ("total_km",     "Total transport distance (km)",    "km",     "tab:blue"),
    ("total_tkm",    "Total tonne-kilometres (tkm)",      "tkm",    "tab:green"),
    ("total_trips",  "Total trips",                       "Trips",  "tab:red"),
    ("total_tonnes", "Total tonnes moved",                "Tonnes", "tab:orange"),
]
for ax, (col, title, ylabel, color) in zip(axes.flatten(), kpi_plots):
    ax.plot(kpis["year"], kpis[col], marker="o", color="black")
    ax.set(title=title, xlabel="Year", ylabel=ylabel)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(f"test_model/{SCENARIO}/kpi_charts_basic.png", dpi=150)
plt.show()



In [ ]:
# =============================================================================
# 11. KPI CHARTS
# =============================================================================

FLOW_LINE_COLORS = FLOW_COLORS.copy()

# Rebuild totals from the active flows_df here as well, so this cell works
# even if the basic KPI cell was not rerun immediately before it.
plot_flows = flows_df.copy()
plot_flows["year"] = plot_flows["year"].astype(str)
for col in ["km", "tkm", "trips", "tonnes"]:
    plot_flows[col] = pd.to_numeric(plot_flows[col], errors="coerce").fillna(0)

kpis = (plot_flows.groupby("year", as_index=False)
        .agg(total_km=("km", "sum"),
             total_tkm=("tkm", "sum"),
             total_trips=("trips", "sum"),
             total_tonnes=("tonnes", "sum"))
        .sort_values("year"))

# per-flow breakdown for each KPI metric
kpi_by_flow = (plot_flows.groupby(["year", "flow"])
               .agg(total_km=("km", "sum"),
                    total_tkm=("tkm", "sum"),
                    total_trips=("trips", "sum"),
                    total_tonnes=("tonnes", "sum"))
               .reset_index())

kpi_plots = [
    ("total_km",     "Total transport distance (km)", "km"),
    ("total_tkm",    "Total tonne-kilometres (tkm)",   "tkm"),
    ("total_trips",  "Total trips",                    "Trips"),
    ("total_tonnes", "Total tonnes moved",             "Tonnes"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, (col, title, ylabel) in zip(axes.flatten(), kpi_plots):
    for flow, color in FLOW_LINE_COLORS.items():
        subset = kpi_by_flow[kpi_by_flow["flow"] == flow]
        if len(subset) > 0:
            ax.plot(subset["year"], subset[col],
                    marker="o", markersize=4,
                    linewidth=1.5, color=color,
                    label=FLOW_LABELS.get(flow, flow.replace("_", " ").title()))

    # total line (thick black) always rebuilt from the same flows_df
    ax.plot(kpis["year"], kpis[col],
            marker="o", markersize=5,
            linewidth=1.5, color="black",
            label="Total", zorder=5)

    ax.set(title=title, xlabel="Year", ylabel=ylabel)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=8, frameon=False)

plt.tight_layout()
plt.savefig(f"test_model/{SCENARIO}/kpi_charts.png", dpi=150)
plt.show()



In [ ]:
# =============================================================================
# 12. MATERIAL FLOW BAR CHART
# =============================================================================

#OLD COLORS - Keep it here
#FLOW_COLORS = {
#    "incoming_new":         "#a6cee3",
#    "incoming_reused":      "#fdae61",
#    "outgoing_reuse":       "#d95f0e",
#    "outgoing_processing":  "#bdbdbd",
#}

FLOW_ORDER = ["incoming_new", "incoming_reused", "outgoing_reuse", "outgoing_processing"]

mat_break = (flows_df.groupby(["year", "flow"], as_index=False)
                    .agg(tonnes=("tonnes", "sum")))

pivot = (mat_break.pivot_table(index="year", columns="flow", values="tonnes",
                                aggfunc="sum", fill_value=0)
                  .reindex(columns=FLOW_ORDER, fill_value=0))

ax = pivot.plot(kind="bar", stacked=True, figsize=(10, 6),
                color=[FLOW_COLORS[c] for c in FLOW_ORDER])
ax.set(title="Material flows per year", xlabel="Year", ylabel="Tonnes")
ax.legend([FLOW_LABELS[c] for c in FLOW_ORDER], title="Flow type", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"test_model/{SCENARIO}/material_flows.png", dpi=150)
plt.show()

### 12b. Material flow composition pie grid (%)
- Inline plotting block stored directly in this notebook.
- Shows the share of recorded flow tonnes by year and material for the active scenario.


In [ ]:
# =============================================================================
# 12b. MATERIAL FLOW COMPOSITION PIE GRID (%)
# =============================================================================

from pathlib import Path
from IPython.display import Image, display

PIE_FLOW_ORDER = [
    "incoming_new",
    "incoming_reused",
    "outgoing_reuse",
    "outgoing_processing",
]
PIE_FLOW_LABELS = {
    "incoming_new": "Incoming - New",
    "incoming_reused": "Incoming - Reused",
    "outgoing_reuse": "Outgoing - Reuse",
    "outgoing_processing": "Outgoing - Processing",
}
PIE_FLOW_COLORS = {flow: FLOW_COLORS[flow] for flow in PIE_FLOW_ORDER}

scenario_dir = Path("test_model") / SCENARIO
material_breakdown_path = scenario_dir / "logistics_material_breakdown.csv"
pie_grid_path = scenario_dir / "material_flow_pie_grid.png"

material_breakdown_df = pd.read_csv(material_breakdown_path)
required_cols = {"year", "material", "flow", "tonnes"}
missing_cols = required_cols.difference(material_breakdown_df.columns)
if missing_cols:
    raise ValueError(f"Missing columns in {material_breakdown_path}: {sorted(missing_cols)}")

material_breakdown_df["year"] = material_breakdown_df["year"].astype(str)
material_breakdown_df["tonnes"] = pd.to_numeric(material_breakdown_df["tonnes"], errors="coerce").fillna(0)
material_breakdown_df = material_breakdown_df[material_breakdown_df["flow"].isin(PIE_FLOW_ORDER)].copy()

all_materials_df = material_breakdown_df.groupby(["year", "flow"], as_index=False)["tonnes"].sum()
all_materials_df["material"] = "All materials"
pie_plot_df = pd.concat([material_breakdown_df, all_materials_df], ignore_index=True)

pie_years = sorted(pie_plot_df["year"].unique(), key=lambda value: int(value) if str(value).isdigit() else str(value))
pie_materials = sorted(material_breakdown_df["material"].dropna().unique()) + ["All materials"]

fig, axes = plt.subplots(
    len(pie_materials),
    len(pie_years),
    figsize=(max(3.0 * len(pie_years), 10), 3.0 * len(pie_materials)),
    squeeze=False,
)

for row, material in enumerate(pie_materials):
    for column, year in enumerate(pie_years):
        ax = axes[row, column]
        subset = pie_plot_df[(pie_plot_df["material"] == material) & (pie_plot_df["year"] == year)]
        values = subset.groupby("flow")["tonnes"].sum().reindex(PIE_FLOW_ORDER, fill_value=0).astype(float)

        if values.sum() > 0:
            ax.pie(
                values,
                colors=[PIE_FLOW_COLORS[flow] for flow in PIE_FLOW_ORDER],
                startangle=90,
                counterclock=False,
                autopct=lambda pct: f"{pct:.0f}%" if pct >= 4 else "",
                pctdistance=0.72,
                textprops={"fontsize": 7},
                wedgeprops={"linewidth": 0.5, "edgecolor": "white"},
            )
        else:
            ax.text(0.5, 0.5, "No flow", ha="center", va="center", fontsize=8)
            ax.set_axis_off()

        if row == 0:
            ax.set_title(year, fontsize=10, pad=6)
        if column == 0:
            ax.set_ylabel(str(material).title(), fontsize=10, labelpad=12)

pie_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor=PIE_FLOW_COLORS[flow],
        markeredgecolor="none",
        markersize=9,
        label=PIE_FLOW_LABELS[flow],
    )
    for flow in PIE_FLOW_ORDER
]
fig.legend(
    handles=pie_handles,
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01)
)
fig.suptitle(
    f"{SCENARIO.title()}: material flow composition by year and material\n"
    "Share of recorded flow tonnes",
    fontsize=14,
    y=0.995,
)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
fig.savefig(pie_grid_path, dpi=180, bbox_inches="tight")
plt.close(fig)

print(f"Saved pie grid: {pie_grid_path}")
display(Image(filename=str(pie_grid_path)))


In [ ]:
# =============================================================================
# DEBUG: which flows are missing ORS geometry on the map?
# =============================================================================

import pandas as pd
import geopandas as gpd

def _to_4326(gdf):
    return gdf.copy().to_crs(4326) if gdf.crs is not None and gdf.crs.to_epsg() != 4326 else gdf.copy()

# Build the same route lookup used by the map
route_lookup = {}
for od_key, gdf in (ORS_ROUTES_GDF or {}).items():
    if gdf is None or len(gdf) == 0:
        continue

    gdf_4326 = _to_4326(gdf)
    for _, r in gdf_4326.iterrows():
        origin = str(r["origin_id"]).strip()
        dest = str(r["destination_id"]).strip()

        route_lookup[(od_key, origin, dest)] = r.geometry

        for alias_key in reverse_ors_aliases(od_key):
            route_lookup[(alias_key, dest, origin)] = r.geometry

# Same grouping logic as the map
agg_flows = (
    flows_df
    .groupby(["od_key", "origin_id", "destination_id", "flow"], as_index=False)
    .agg(
        total_trips=("trips", "sum"),
        total_tonnes=("tonnes", "sum"),
        material=("material", "first"),
    )
)

agg_flows["origin_id"] = agg_flows["origin_id"].astype(str).str.strip()
agg_flows["destination_id"] = agg_flows["destination_id"].astype(str).str.strip()

agg_flows["has_geometry"] = agg_flows.apply(
    lambda r: (r["od_key"], r["origin_id"], r["destination_id"]) in route_lookup,
    axis=1,
)

# Summary by flow type
summary = (
    agg_flows.groupby("flow", as_index=False)
    .agg(
        total_rows=("flow", "size"),
        mapped_rows=("has_geometry", "sum"),
    )
)
summary["missing_rows"] = summary["total_rows"] - summary["mapped_rows"]
summary["pct_mapped"] = (100 * summary["mapped_rows"] / summary["total_rows"]).round(1)

print("\n=== Geometry coverage by flow type ===")
print(summary.sort_values("flow").to_string(index=False))

# All missing rows
missing = agg_flows[~agg_flows["has_geometry"]].copy()

print(f"\n=== Total missing geometry rows: {len(missing)} ===")

if len(missing) == 0:
    print("No missing geometry rows.")
else:
    print("\n=== Missing rows by flow type ===")
    for flow_name in ["incoming_new", "incoming_reused", "outgoing_reuse", "outgoing_processing"]:
        part = missing[missing["flow"] == flow_name].copy()

        print(f"\n--- {flow_name}: {len(part)} missing ---")
        if len(part) == 0:
            print("None")
            continue

        print(
            part[
                ["od_key", "origin_id", "destination_id", "material", "total_trips", "total_tonnes"]
            ]
            .sort_values(["origin_id", "destination_id"])
            .head(25)
            .to_string(index=False)
        )

# Optional: full detailed table you can inspect in notebook
missing_geometry_debug = missing[
    ["flow", "od_key", "origin_id", "destination_id", "material", "total_trips", "total_tonnes"]
].sort_values(["flow", "origin_id", "destination_id"]).reset_index(drop=True)

print("\nSaved missing rows to variable: missing_geometry_debug")


In [ ]:
# =============================================================================
# 13. FOLIUM MAP
# =============================================================================

FLOW_LINE_COLORS = FLOW_COLORS.copy()

def build_folium_map(flows_df, ors_routes_gdf, edge_usage_gdf,
                     projects, suppliers, hubs, processing,
                     theme="Esri_WorldGrayCanvas", road_color="#333333",
                     output_html="logistics_map.html"):
    import folium

    def to_4326(gdf):
        return gdf.copy().to_crs(4326) if gdf.crs and gdf.crs.to_epsg() != 4326 else gdf.copy()

    projects   = to_4326(projects)
    suppliers  = to_4326(suppliers)
    hubs       = to_4326(hubs)
    processing = to_4326(processing)
    proc_col   = next((c for c in processing.columns if "name" in c.lower()), processing.columns[0])

    # Esri World Gray Canvas basemap (no API key in this tile URL)
    m = folium.Map(
        location=[52.377, 4.899],
        zoom_start=11,
        tiles=None,
    )
    folium.TileLayer(
        tiles=(
            "https://server.arcgisonline.com/ArcGIS/rest/services/"
            "Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
        ),
        attr="Tiles © Esri",
        name="Esri World Gray Canvas",
    ).add_to(m)
    folium.map.CustomPane("markers", z_index=650).add_to(m)

    # ── Layer 2: flow lines ───────────────────────────────────────────────────
    # build route geometry lookup
    route_lookup = {}
    for od_key, gdf in (ors_routes_gdf or {}).items():
        if gdf is None or len(gdf) == 0:
            continue
        gdf_4326 = to_4326(gdf)
        for _, r in gdf_4326.iterrows():
            origin = str(r["origin_id"]).strip()
            dest = str(r["destination_id"]).strip()
            k = (od_key, origin, dest)
            route_lookup[k] = r.geometry
            for alias_key in reverse_ors_aliases(od_key):
                route_lookup[(alias_key, dest, origin)] = r.geometry

    # one FeatureGroup per flow type (off by default) + all flows grey (on by default)
    flow_groups = {
        flow: folium.FeatureGroup(name=flow.replace("_", " ").title(), show=False)
        for flow in FLOW_LINE_COLORS
    }
    all_flows_group = folium.FeatureGroup(name="All Flows", show=True)

    if len(flows_df) > 0:
        agg_flows = (flows_df
                     .groupby(["od_key", "origin_id", "destination_id", "flow"], as_index=False)
                     .agg(total_trips=("trips", "sum"),
                          total_tonnes=("tonnes", "sum"),
                          material=("material", "first")))

        # combined trips per OD pair across all flow types — drives all flows thickness
        all_flows_trips = (agg_flows
                           .groupby(["od_key", "origin_id", "destination_id"], as_index=False)
                           ["total_trips"].sum()
                           .rename(columns={"total_trips": "combined_trips"}))

        drawn = 0
        missing = 0
        for _, row in agg_flows.iterrows():
            k = (row["od_key"], str(row["origin_id"]).strip(), str(row["destination_id"]).strip())
            geom = route_lookup.get(k)
            if geom is None:
                missing += 1
                continue

            coords = [(lat, lon) for lon, lat in geom.coords]
            color  = FLOW_LINE_COLORS.get(row["flow"], "#999999")
            weight = max(1, min(6, 1 + row["total_trips"] / 20))
            popup  = folium.Popup(
                f"<b>{row['flow']}</b><br>"
                f"Material: {row['material']}<br>"
                f"Trips: {row['total_trips']}<br>"
                f"Tonnes: {row['total_tonnes']:.1f} t",
                max_width=200
            )

            # coloured line → individual flow group
            folium.PolyLine(
                locations=coords, color=color, weight=weight,
                opacity=0.7, popup=popup
            ).add_to(flow_groups[row["flow"]])

            # grey line → all flows group, thickness = combined trips on this OD pair
            combined = all_flows_trips.loc[
                (all_flows_trips["od_key"]        == row["od_key"]) &
                (all_flows_trips["origin_id"]     == row["origin_id"]) &
                (all_flows_trips["destination_id"]== row["destination_id"]),
                "combined_trips"
            ].values[0]
            weight_all = max(1, min(6, 1 + combined / 20))
            folium.PolyLine(
                locations=coords, color="#aaaaaa", weight=weight_all,
                opacity=0.5, popup=popup
            ).add_to(all_flows_group)

            drawn += 1

        print(f"  Route lines drawn: {drawn}  |  missing ORS geometry: {missing}")

    # add flow groups to map
    all_flows_group.add_to(m)
    for fg in flow_groups.values():
        fg.add_to(m)

    # ── Layer 3: point markers (kept above route lines) ───────────────────────
    points_group = folium.FeatureGroup(name="Locations", show=True)

    for _, r in projects.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=3, color="orange", weight=0.8,
            fill=True, fill_color="white", fill_opacity=1.0,
            popup=folium.Popup(f"Project: {r.get('Projnaam', '')}", max_width=200),
            pane="markers"
        ).add_to(points_group)

    for _, r in suppliers.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=7, color="#1f78b4", weight=1.5,
            fill=True, fill_color="#1f78b4", fill_opacity=0.9,
            popup=folium.Popup(f"Supplier: {r.get('name', '')}", max_width=200),
            pane="markers"
        ).add_to(points_group)

    for _, r in hubs.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=8, color="#d73027", weight=1.5,
            fill=True, fill_color="#d73027", fill_opacity=0.9,
            popup=folium.Popup(f"Hub: {r.get('name', '')}", max_width=200),
            pane="markers"
        ).add_to(points_group)

    for _, r in processing.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=6, color="#ff7f00", weight=1.5,
            fill=True, fill_color="#ff7f00", fill_opacity=0.9,
            popup=folium.Popup(f"Processing: {r.get(proc_col, '')}", max_width=200),
            pane="markers"
        ).add_to(points_group)

    points_group.add_to(m)

    # ── Legend ────────────────────────────────────────────────────────────────
    legend_html = """
    <div style="position:fixed; bottom:30px; left:30px; z-index:1000;
                background:rgba(0,0,0,0.75); padding:12px 16px; border-radius:8px;
                font-family:sans-serif; font-size:12px; color:white; line-height:1.8;">
        <b>Flow types</b><br>
        <span style="color:#aaaaaa">——</span> All flows<br>
        <span style="color:#4575b4">——</span> Incoming new (supplier)<br>
        <span style="color:#f46d43">——</span> Incoming reused (hub)<br>
        <span style="color:#d73027">——</span> Outgoing to hub<br>
        <span style="color:#878787">——</span> Outgoing to processing<br>
        <hr style="border-color:#555; margin:6px 0">
        <b>Route thickness</b><br>
        Thicker line = more vehicles / trips<br>
        <hr style="border-color:#555; margin:6px 0">
        <span style="color:white">● </span> Project &nbsp;
        <span style="color:#1f78b4">● </span> Supplier<br>
        <span style="color:#d73027">● </span> Hub &nbsp;
        <span style="color:#ff7f00">● </span> Processing
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    folium.LayerControl().add_to(m)
    m.save(output_html)
    print(f"Map saved: {output_html}")
    return m


In [ ]:
# ── Reload geodata & build map ────────────────────────────────────────────────

projects_4326   = gpd.read_file("data/cleaned/projects.gpkg")
suppliers_4326  = gpd.read_file("data/cleaned/suppliers_gdf.gpkg")
hubs_4326       = gpd.read_file("data/processed/hubs_storage_current.gpkg")
processing_4326 = gpd.read_file("data/cleaned/processing_locations_gdf_clean.gpkg")

print("Building Esri World Gray Canvas map...")

build_folium_map(
    flows_df       = flows_df,
    ors_routes_gdf = ORS_ROUTES_GDF,
    edge_usage_gdf = None,
    projects       = projects_4326.copy(),
    suppliers      = suppliers_4326.copy(),
    hubs           = hubs_4326.copy(),
    processing     = processing_4326.copy(),
    theme          = "Esri_WorldGrayCanvas",
    road_color     = "#333333",
    output_html    = f"test_model/{SCENARIO}/logistics_map_light_flows3.html",
)

In [ ]:
# =============================================================================
# 14. EMISSIONS CHARTS (CO2, NOx, PM2.5, PM10)
# =============================================================================

print("Emission factors loaded from vehicles_info_basic.csv:", {k: v for k, v in VEHICLE_LIBRARY.items()})

# Add emission columns to flows_df using vehicle + fuel specific factors.
flows_df["co2_tCO2"] = flows_df.apply(
    lambda r: VEHICLE_LIBRARY.get((r["vehicle"], r["fuel"]), {}).get("co2_tCO2_per_km", 0) * r["km"], axis=1
)
flows_df["nox_g"] = flows_df.apply(
    lambda r: VEHICLE_LIBRARY.get((r["vehicle"], r["fuel"]), {}).get("nox_g_per_km", 0) * r["km"], axis=1
)
flows_df["pm25_g"] = flows_df.apply(
    lambda r: VEHICLE_LIBRARY.get((r["vehicle"], r["fuel"]), {}).get("pm25_g_per_km", 0) * r["km"], axis=1
)
flows_df["pm10_g"] = flows_df.apply(
    lambda r: VEHICLE_LIBRARY.get((r["vehicle"], r["fuel"]), {}).get("pm10_g_per_km", 0) * r["km"], axis=1
)

em_kpis = (flows_df.groupby("year")
           .agg(total_co2 =("co2_tCO2", "sum"),
                total_nox =("nox_g",    "sum"),
                total_pm25=("pm25_g",   "sum"),
                total_pm10=("pm10_g",   "sum"))
           .reset_index()
           .sort_values("year"))

em_kpis["total_nox"]  = em_kpis["total_nox"]  / 1000
em_kpis["total_pm25"] = em_kpis["total_pm25"] / 1000
em_kpis["total_pm10"] = em_kpis["total_pm10"] / 1000

emissions_config = [
    ("total_co2",  "CO₂ (tCO₂)",    EMISSIONS_COLORS["total_co2"]),
    ("total_nox",  "NOx (kg)",      EMISSIONS_COLORS["total_nox"]),
    ("total_pm25", "PM2.5 (kg)",    EMISSIONS_COLORS["total_pm25"]),
    ("total_pm10", "PM10 (kg)",     EMISSIONS_COLORS["total_pm10"]),
]
fig, axes = plt.subplots(4, 2, figsize=(14, 18))

for row_axes, (col, label, color) in zip(axes, emissions_config):
    ax_bar, ax_cum = row_axes
    years_str  = em_kpis["year"].astype(str)
    values     = em_kpis[col]
    cumulative = values.cumsum()

    ax_bar.bar(years_str, values, color=color, alpha=0.8, edgecolor="white", linewidth=0.5)
    ax_bar.set(title=f"{label} per year", xlabel="Year", ylabel=label)
    ax_bar.spines[["top", "right"]].set_visible(False)
    for bar, val in zip(ax_bar.patches, values):
        ax_bar.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + values.max() * 0.01 if len(values) and values.max() > 0 else 0,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=7)

    ax_cum.plot(years_str, cumulative, marker="o", color=color, linewidth=2.5, markersize=6)
    ax_cum.fill_between(range(len(em_kpis)), cumulative, alpha=0.15, color=color)
    ax_cum.set_xticks(range(len(em_kpis)))
    ax_cum.set_xticklabels(years_str)
    ax_cum.set(title=f"{label} cumulative", xlabel="Year", ylabel=f"{label} (cumulative)")
    ax_cum.spines[["top", "right"]].set_visible(False)

plt.suptitle("Emissions — all flows (diesel + electric vehicles)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"test_model/{SCENARIO}/emissions_charts.png", dpi=150, bbox_inches="tight")
plt.show()

# Export the emissions-enriched flows and annual emissions for the dashboard.
flows_df.to_csv(f"test_model/{SCENARIO}/logistics_flows_per_project_year.csv", index=False)
em_kpis.to_csv(f"test_model/{SCENARIO}/logistics_emissions_per_year.csv", index=False)
print(f"Saved emissions-enriched outputs for {SCENARIO}.")


### SCENARIOS (when all saved)

- add nice visualisation with index as in bimzec.nl 
- and visualisation between the number transport km and trips
- this can be a separate notebook
- separate notebook for equity analysis, connection with population + impact buffer

In [ ]:
# =============================================================================
# 15. SCENARIO COMPARISON
# =============================================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

SCENARIO_STYLES = {
    "baseline": {"label": "Baseline",              "color": "#f5c381", "ls": "-"},
    "linear":   {"label": "Linear (no reuse)",     "color": "#afadac", "ls": "--"},
    "circular": {"label": "Circular (high reuse)", "color": "#fa930d", "ls": "-."},
}

LEGACY_VEHICLE_MAP = {
    "small truck": "Van",
    "small_truck": "Van",
    "bestelauto": "Van",
    "van": "Van",
    "medium truck": "Medium truck",
    "medium_truck": "Medium truck",
    "bakwagen": "Medium truck",
    "heavy truck": "Heavy truck",
    "large truck": "Heavy truck",
    "large_truck": "Heavy truck",
    "trekker opleggercombinatie": "Heavy truck",
    "trekker-opleggercombinatie": "Heavy truck",
}

def normalize_vehicle_for_comparison(value):
    key = normalize_text(value)
    return LEGACY_VEHICLE_MAP.get(key)

def normalize_fuel_for_comparison(value):
    key = normalize_text(value)
    return key if key in {"diesel", "electric"} else "diesel"

def kpis_from_flows_df(flows):
    work = flows.copy()
    work["year"] = work["year"].astype(str)
    for col in ["km", "tkm", "trips", "tonnes"]:
        work[col] = pd.to_numeric(work.get(col, 0), errors="coerce").fillna(0)
    return (work.groupby("year", as_index=False)
            .agg(total_km=("km", "sum"),
                 total_tkm=("tkm", "sum"),
                 total_trips=("trips", "sum"),
                 total_tonnes=("tonnes", "sum"))
            .sort_values("year"))

def emissions_from_flows_df(flows):
    work = flows.copy()
    work["km"] = pd.to_numeric(work.get("km", 0), errors="coerce").fillna(0)
    vehicle_series = work["vehicle"] if "vehicle" in work.columns else pd.Series([None] * len(work), index=work.index)
    work["vehicle_norm"] = vehicle_series.apply(normalize_vehicle_for_comparison)
    if "fuel" in work.columns:
        work["fuel_norm"] = work["fuel"].apply(normalize_fuel_for_comparison)
    else:
        work["fuel_norm"] = "diesel"
    work["co2_tCO2"] = work.apply(
        lambda r: VEHICLE_LIBRARY.get((r["vehicle_norm"], r["fuel_norm"]), {}).get("co2_tCO2_per_km", 0) * r["km"],
        axis=1,
    )
    return (work.groupby("year", as_index=False)
            .agg(total_co2=("co2_tCO2", "sum"))
            .sort_values("year"))

kpi_all = {}
em_all  = {}
comparison_rows = []

for scenario in SCENARIO_STYLES:
    kpi_path = f"test_model/{scenario}/logistics_kpis_per_year.csv"
    flows_path = f"test_model/{scenario}/logistics_flows_per_project_year.csv"

    total_co2 = np.nan
    if os.path.exists(flows_path):
        flows = pd.read_csv(flows_path)
        if len(flows) > 0:
            kpi_df = kpis_from_flows_df(flows)
            kpi_all[scenario] = kpi_df
            em_df = emissions_from_flows_df(flows)
            em_all[scenario] = em_df
            total_co2 = em_df["total_co2"].sum()
        elif os.path.exists(kpi_path):
            kpi_all[scenario] = pd.read_csv(kpi_path)
    elif os.path.exists(kpi_path):
        kpi_all[scenario] = pd.read_csv(kpi_path)
    else:
        print(f"  Missing scenario outputs for '{scenario}' — run that scenario first")
        continue

    if scenario not in kpi_all:
        continue

    kpi_df = kpi_all[scenario]
    comparison_rows.append({
        "scenario": SCENARIO_STYLES[scenario]["label"],
        "total_km": kpi_df["total_km"].sum() if "total_km" in kpi_df.columns else np.nan,
        "total_tkm": kpi_df["total_tkm"].sum() if "total_tkm" in kpi_df.columns else np.nan,
        "total_trips": kpi_df["total_trips"].sum() if "total_trips" in kpi_df.columns else np.nan,
        "total_tonnes": kpi_df["total_tonnes"].sum() if "total_tonnes" in kpi_df.columns else np.nan,
        "total_co2_tCO2": total_co2,
    })

kpi_metrics = [
    ("total_km",     "Total transport distance (km)", "km"),
    ("total_tkm",    "Total tonne-kilometres (tkm)",  "tkm"),
    ("total_trips",  "Total trips",                   "Trips"),
    ("total_tonnes", "Total tonnes moved",            "Tonnes"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (col, title, ylabel) in zip(axes.flatten(), kpi_metrics):
    for scenario, kpi_df in kpi_all.items():
        s = SCENARIO_STYLES[scenario]
        if col in kpi_df.columns:
            ax.plot(kpi_df["year"], kpi_df[col], marker="o", markersize=5, linewidth=2, color=s["color"], linestyle=s["ls"], label=s["label"])
    ax.set(title=title, xlabel="Year", ylabel=ylabel)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=8, frameon=False)

plt.suptitle("Scenario comparison — KPIs", fontsize=14)
plt.tight_layout()
plt.savefig("test_model/comparison_kpis.png", dpi=150, bbox_inches="tight")
plt.show()

if em_all:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax1 = axes[0]
    years = sorted({y for df in em_all.values() for y in df["year"]})
    x = np.arange(len(years))
    width = 0.25
    for i, scenario in enumerate(em_all):
        em_df = em_all[scenario]
        s = SCENARIO_STYLES[scenario]
        vals = [em_df.loc[em_df["year"] == y, "total_co2"].sum() for y in years]
        ax1.bar(x + i * width, vals, width, label=s["label"], color=s["color"], alpha=0.85)
    ax1.set_xticks(x + width)
    ax1.set_xticklabels([str(y) for y in years], rotation=45)
    ax1.set(title="CO₂ per year by scenario", xlabel="Year", ylabel="tCO₂")
    ax1.spines[["top", "right"]].set_visible(False)
    ax1.legend(fontsize=8, frameon=False)

    ax2 = axes[1]
    last_year_labels = None
    for scenario, em_df in em_all.items():
        s = SCENARIO_STYLES[scenario]
        em_sorted = em_df.sort_values("year")
        cum = em_sorted["total_co2"].cumsum()
        year_labels = em_sorted["year"].astype(str).tolist()
        ax2.plot(year_labels, cum, marker="o", linewidth=2.5, markersize=6, color=s["color"], linestyle=s["ls"], label=s["label"])
        ax2.fill_between(range(len(em_sorted)), cum, alpha=0.08, color=s["color"])
        last_year_labels = year_labels
    if last_year_labels is not None:
        ax2.set_xticks(range(len(last_year_labels)))
        ax2.set_xticklabels(last_year_labels)
    ax2.set(title="Cumulative CO₂ by scenario", xlabel="Year", ylabel="tCO₂ (cumulative)")
    ax2.spines[["top", "right"]].set_visible(False)
    ax2.legend(fontsize=8, frameon=False)

    plt.suptitle("Scenario comparison — CO₂ emissions", fontsize=14)
    plt.tight_layout()
    plt.savefig("test_model/comparison_co2.png", dpi=150, bbox_inches="tight")
    plt.show()

comparison_df = pd.DataFrame(comparison_rows)
if len(comparison_df) > 0:
    print(comparison_df.to_string(index=False))
    comparison_df.to_csv("test_model/scenario_comparison.csv", index=False)
    print("Saved: test_model/comparison_kpis.png, comparison_co2.png, scenario_comparison.csv")



### Baseline sensitivity analysis
- This section keeps the three main scenarios unchanged and tests one parameter family at a time around the baseline case.
- Current baseline values are the low case; each parameter then gets two improved levels.


In [ ]:
# =============================================================================
# 16. BASELINE SENSITIVITY ANALYSIS
# =============================================================================

import os

BASELINE_SETTINGS = SCENARIOS["baseline"]
BASELINE_REUSE = (pd.read_csv(BASELINE_SETTINGS["reuse_rates_path"])
                    .rename(columns={"Pavement material": "material",
                                     "Incoming reuse (%)": "incoming_reuse_pct",
                                     "Outgoing reuse (%)": "outgoing_reuse_pct"})
                    .assign(material_norm=lambda d: d["material"].apply(norm_material))
                    [["material_norm", "incoming_reuse_pct", "outgoing_reuse_pct"]])

BASELINE_SENSITIVITY_CASES = [
    {"case": "baseline_ref", "label": "Baseline current",      "family": "Reference",            "onsite_direct_reuse": 0.05, "ingoing_load_factor": 0.875, "outgoing_load_factor": 0.60, "ev_level": "current"},
    {"case": "onsite_mid",   "label": "Onsite reuse 15%",     "family": "Onsite reuse",         "onsite_direct_reuse": 0.15, "ingoing_load_factor": 0.875, "outgoing_load_factor": 0.60, "ev_level": "current"},
    {"case": "onsite_high",  "label": "Onsite reuse 30%",     "family": "Onsite reuse",         "onsite_direct_reuse": 0.30, "ingoing_load_factor": 0.875, "outgoing_load_factor": 0.60, "ev_level": "current"},
    {"case": "inload_mid",   "label": "Ingoing load 90%",     "family": "Ingoing load factor",  "onsite_direct_reuse": 0.05, "ingoing_load_factor": 0.90,  "outgoing_load_factor": 0.60, "ev_level": "current"},
    {"case": "inload_high",  "label": "Ingoing load 95%",     "family": "Ingoing load factor",  "onsite_direct_reuse": 0.05, "ingoing_load_factor": 0.95,  "outgoing_load_factor": 0.60, "ev_level": "current"},
    {"case": "outload_mid",  "label": "Outgoing load 75%",    "family": "Outgoing load factor", "onsite_direct_reuse": 0.05, "ingoing_load_factor": 0.875, "outgoing_load_factor": 0.75, "ev_level": "current"},
    {"case": "outload_high", "label": "Outgoing load 90%",    "family": "Outgoing load factor", "onsite_direct_reuse": 0.05, "ingoing_load_factor": 0.875, "outgoing_load_factor": 0.90, "ev_level": "current"},
    {"case": "ev_mid",       "label": "EV improved",          "family": "Fleet electrification", "onsite_direct_reuse": 0.05, "ingoing_load_factor": 0.875, "outgoing_load_factor": 0.60, "ev_level": "improved"},
    {"case": "ev_high",      "label": "EV high",              "family": "Fleet electrification", "onsite_direct_reuse": 0.05, "ingoing_load_factor": 0.875, "outgoing_load_factor": 0.60, "ev_level": "high"},
]

SENSITIVITY_COLORS = {
    "Reference": "#4b5563",
    "Onsite reuse": "#1f9d8a",
    "Ingoing load factor": "#4575b4",
    "Outgoing load factor": "#f46d43",
    "Fleet electrification": "#7b3294",
}

sens_proj_rows = [
    {"project_key": idx, "Projnaam": r["Projnaam"],
     "Aannemer": r.get("Aannemer"),
     "Stadsdeel": r.get("Stadsdeel"),
     "material_norm": m, "tonnes": float(t),
     "geometry": r["geometry"], "year": r["year_bucket"]}
    for idx, r in projects.iterrows()
    for m, t in split_materials(r)
]
base_proj_df = gpd.GeoDataFrame(sens_proj_rows, geometry="geometry", crs=projects.crs)
base_proj_df = base_proj_df.merge(BASELINE_REUSE, on="material_norm", how="left")
base_proj_df["incoming_reuse_pct"] = base_proj_df["incoming_reuse_pct"].fillna(0)
base_proj_df["outgoing_reuse_pct"] = base_proj_df["outgoing_reuse_pct"].fillna(0)

baseline_hubs = BASELINE_SETTINGS["active_hubs"]

def hub_for_baseline(mat):
    pt, name = (beentjes_pt, beentjes_name) if mat in STONE_MATS else (stads_pt, stads_name)
    if baseline_hubs != ["all"] and name not in baseline_hubs:
        return None, None
    return pt, name


def sensitivity_logistics_mix(flow_type: str, stadsdeel: str, ev_level: str):
    centrum = is_centrum(stadsdeel)
    if ev_level == "current":
        return logistics_mix(flow_type, stadsdeel)

    if flow_type == "incoming_new":
        if centrum:
            if ev_level == "improved":
                return [
                    {"vehicle": "Heavy truck", "fuel": "electric", "share": 0.45},
                    {"vehicle": "Heavy truck", "fuel": "diesel", "share": 0.05},
                    {"vehicle": "Medium truck", "fuel": "electric", "share": 0.45},
                    {"vehicle": "Medium truck", "fuel": "diesel", "share": 0.05},
                ]
            return [
                {"vehicle": "Heavy truck", "fuel": "electric", "share": 0.50},
                {"vehicle": "Medium truck", "fuel": "electric", "share": 0.50},
            ]
        if ev_level == "improved":
            return [
                {"vehicle": "Heavy truck", "fuel": "diesel", "share": 0.75},
                {"vehicle": "Heavy truck", "fuel": "electric", "share": 0.25},
            ]
        return [
            {"vehicle": "Heavy truck", "fuel": "diesel", "share": 0.50},
            {"vehicle": "Heavy truck", "fuel": "electric", "share": 0.50},
        ]

    if centrum:
        if ev_level == "improved":
            return [
                {"vehicle": "Van", "fuel": "electric", "share": 0.54},
                {"vehicle": "Van", "fuel": "diesel", "share": 0.06},
                {"vehicle": "Medium truck", "fuel": "electric", "share": 0.36},
                {"vehicle": "Medium truck", "fuel": "diesel", "share": 0.04},
            ]
        return [
            {"vehicle": "Van", "fuel": "electric", "share": 0.60},
            {"vehicle": "Medium truck", "fuel": "electric", "share": 0.40},
        ]

    if ev_level == "improved":
        return [
            {"vehicle": "Medium truck", "fuel": "electric", "share": 0.455},
            {"vehicle": "Medium truck", "fuel": "diesel", "share": 0.245},
            {"vehicle": "Van", "fuel": "electric", "share": 0.195},
            {"vehicle": "Van", "fuel": "diesel", "share": 0.105},
        ]
    return [
        {"vehicle": "Medium truck", "fuel": "electric", "share": 0.595},
        {"vehicle": "Medium truck", "fuel": "diesel", "share": 0.105},
        {"vehicle": "Van", "fuel": "electric", "share": 0.255},
        {"vehicle": "Van", "fuel": "diesel", "share": 0.045},
    ]


def run_baseline_sensitivity_case(case_cfg: dict):
    proj_case = base_proj_df.copy()
    proj_case["tonnes_onsite_reuse"] = proj_case["tonnes"] * case_cfg["onsite_direct_reuse"]
    proj_case["tonnes_for_inbound"] = proj_case["tonnes"] - proj_case["tonnes_onsite_reuse"]
    proj_case["tonnes_reused_in"] = proj_case["tonnes_for_inbound"] * (proj_case["incoming_reuse_pct"] / 100)
    proj_case["tonnes_new_in"] = proj_case["tonnes_for_inbound"] - proj_case["tonnes_reused_in"]
    proj_case["tonnes_breakage"] = proj_case["tonnes"] * BREAKAGE_PCT
    proj_case["tonnes_out_for_reuse"] = proj_case["tonnes"] * (proj_case["outgoing_reuse_pct"] / 100)
    proj_case["tonnes_out_for_processing"] = np.maximum(
        proj_case["tonnes"] - proj_case["tonnes_out_for_reuse"], proj_case["tonnes_breakage"]
    )

    supplier_rng_local = np.random.default_rng(2026)
    processing_rng_local = np.random.default_rng(2027)

    def select_supplier_local(mat: str):
        cand = supplier_candidates(mat)
        if len(cand) == 0:
            return None, None
        if mat == "gebakken steen":
            chosen_company = supplier_rng_local.choice(["vandersanden", "wienerberger"], p=[0.7, 0.3])
            company_cand = cand[cand["company_norm"] == chosen_company]
            if len(company_cand) > 0:
                cand = company_cand
        elif mat in {"betontegel", "betonsteen"}:
            company_cand = cand[cand["company_norm"] == "struyk verwo"]
            if len(company_cand) > 0:
                cand = company_cand
        idx = int(supplier_rng_local.choice(cand.index.to_numpy()))
        return cand.geometry.loc[idx], cand.loc[idx]

    def choose_processing_destination_local(material: str, contractor: str):
        table = COMPLEX_REUSE_TABLES.get(material)
        if table is None:
            return None
        contractor_norm = canonical_contractor_name(contractor)
        matches = table[table["contractor_norm"] == contractor_norm]
        if len(matches) == 0:
            return None
        row = matches.iloc[0]
        options = []
        for loc_col, rate_col, channel in PROCESSING_COLUMN_RULES:
            location_name = row.get(loc_col)
            rate = pd.to_numeric(row.get(rate_col), errors="coerce")
            if pd.isna(location_name) or pd.isna(rate) or float(rate) <= 0:
                continue
            proc_row = processing_lookup.get(normalize_text(location_name))
            if proc_row is None:
                continue
            options.append({
                "channel": channel,
                "name": _id_str(proc_row[proc_name_col]),
                "geometry": proc_row.geometry,
            })
        if not options:
            return None
        return options[int(processing_rng_local.integers(0, len(options)))]

    def build_sensitivity_flow_records(r, flow_type, od_key, orig_type, orig_name, orig_pt, dest_type, dest_name, dest_pt, tonnes, channel=None):
        d, dsrc = leg_distance_km(od_key, orig_name, dest_name, orig_pt, dest_pt)
        load_factor = case_cfg["ingoing_load_factor"] if flow_type in {"incoming_new", "incoming_reused"} else case_cfg["outgoing_load_factor"]
        mix = sensitivity_logistics_mix(flow_type, r.get("Stadsdeel"), case_cfg["ev_level"])
        allocated_parts = allocate_trip_mix(tonnes, mix, load_factor)
        subrecords = []
        for part in allocated_parts:
            tonnes_share = part["tonnes"]
            trips = part["trips"]
            subrecords.append({
                "year": r["year"],
                "project_key": r["project_key"],
                "project_name": _id_str(r.get("Projnaam")),
                "contractor": _id_str(r.get("Aannemer")),
                "stadsdeel": _id_str(r.get("Stadsdeel")),
                "material": r["material_norm"],
                "flow": flow_type,
                "od_key": od_key,
                "origin_id": orig_name,
                "destination_id": dest_name,
                "origin": orig_type,
                "origin_name": orig_name,
                "destination": dest_type,
                "channel": channel,
                "fuel": part["fuel"],
                "vehicle": part["vehicle"],
                "vehicle_class": part["vehicle"],
                "vehicle_source_name": VEHICLE_NAME_MAP[part["vehicle"]],
                "mix_share": part["share"],
                "capacity_t": part["capacity_t"],
                "load_factor": load_factor,
                "tonnes": tonnes_share,
                "trips": trips,
                "distance_source": dsrc,
                "km": d * 2 * trips,
                "tkm": d * 2 * tonnes_share,
            })
        return subrecords

    records = []
    for _, r in proj_case.iterrows():
        proj_pt = r["geometry"]
        proj_name = _id_str(r.get("Projnaam"))
        mat = r["material_norm"]
        contractor = r.get("Aannemer")
        hub_pt, hub_name = hub_for_baseline(mat)

        if r["tonnes_reused_in"] > 0 and hub_pt is not None:
            records.extend(build_sensitivity_flow_records(r, "incoming_reused", "hubs__projects", "hub", hub_name, hub_pt, "project", proj_name, proj_pt, r["tonnes_reused_in"]))
        if r["tonnes_new_in"] > 0:
            sgeom, srow = select_supplier_local(mat)
            if sgeom is not None:
                sname = _id_str(srow.get("name", "supplier"))
                records.extend(build_sensitivity_flow_records(r, "incoming_new", "suppliers__projects", "supplier", sname, sgeom, "project", proj_name, proj_pt, r["tonnes_new_in"]))
        if r["tonnes_out_for_reuse"] > 0 and hub_pt is not None:
            records.extend(build_sensitivity_flow_records(r, "outgoing_reuse", "projects__hubs", "project", proj_name, proj_pt, "hub", hub_name, hub_pt, r["tonnes_out_for_reuse"]))
        if r["tonnes_out_for_processing"] > 0 and len(processing) > 0:
            processing_choice = choose_processing_destination_local(mat, contractor)
            if processing_choice is None:
                fallback_idx = int(processing_rng_local.integers(0, len(processing)))
                proc_row = processing.iloc[fallback_idx]
                proc_pt = proc_row.geometry
                proc_name = _id_str(proc_row[proc_name_col])
                channel = "fallback_random"
            else:
                proc_pt = processing_choice["geometry"]
                proc_name = processing_choice["name"]
                channel = processing_choice["channel"]
            records.extend(build_sensitivity_flow_records(r, "outgoing_processing", "projects__processing", "project", proj_name, proj_pt, "processing", proc_name, proc_pt, r["tonnes_out_for_processing"], channel=channel))

    flows = pd.DataFrame(records)
    if len(flows) == 0:
        return flows, pd.DataFrame(), pd.DataFrame(), {
            "case": case_cfg["case"],
            "label": case_cfg["label"],
            "family": case_cfg["family"],
            "onsite_direct_reuse": case_cfg["onsite_direct_reuse"],
            "ingoing_load_factor": case_cfg["ingoing_load_factor"],
            "outgoing_load_factor": case_cfg["outgoing_load_factor"],
            "ev_level": case_cfg["ev_level"],
            "total_km": 0.0,
            "total_tkm": 0.0,
            "total_trips": 0.0,
            "total_tonnes": 0.0,
            "total_co2_tCO2": 0.0,
        }

    kpi_df = kpis_from_flows_df(flows)
    em_df = emissions_from_flows_df(flows).rename(columns={"total_co2": "total_co2_tCO2"})
    totals = {
        "case": case_cfg["case"],
        "label": case_cfg["label"],
        "family": case_cfg["family"],
        "onsite_direct_reuse": case_cfg["onsite_direct_reuse"],
        "ingoing_load_factor": case_cfg["ingoing_load_factor"],
        "outgoing_load_factor": case_cfg["outgoing_load_factor"],
        "ev_level": case_cfg["ev_level"],
        "total_km": float(kpi_df["total_km"].sum()),
        "total_tkm": float(kpi_df["total_tkm"].sum()),
        "total_trips": float(kpi_df["total_trips"].sum()),
        "total_tonnes": float(kpi_df["total_tonnes"].sum()),
        "total_co2_tCO2": float(em_df["total_co2_tCO2"].sum()),
    }
    return flows, kpi_df, em_df, totals

sensitivity_totals = []
sensitivity_kpis = {}
sensitivity_emissions = {}

for case_cfg in BASELINE_SENSITIVITY_CASES:
    flows_case, kpi_case, em_case, totals_case = run_baseline_sensitivity_case(case_cfg)
    sensitivity_totals.append(totals_case)
    sensitivity_kpis[case_cfg["case"]] = kpi_case
    sensitivity_emissions[case_cfg["case"]] = em_case
    print(f"  {case_cfg['label']}: trips={totals_case['total_trips']:.0f}, km={totals_case['total_km']:.0f}, CO2={totals_case['total_co2_tCO2']:.1f} tCO2")

sensitivity_df = pd.DataFrame(sensitivity_totals)
family_order = ["Reference", "Onsite reuse", "Ingoing load factor", "Outgoing load factor", "Fleet electrification"]
sensitivity_df["family"] = pd.Categorical(sensitivity_df["family"], categories=family_order, ordered=True)
sensitivity_df = sensitivity_df.sort_values(["family", "label"]).reset_index(drop=True)

baseline_row = sensitivity_df[sensitivity_df["case"] == "baseline_ref"].iloc[0]
for metric in ["total_km", "total_trips", "total_co2_tCO2"]:
    sensitivity_df[f"delta_{metric}_pct"] = 100 * (sensitivity_df[metric] - baseline_row[metric]) / baseline_row[metric]

os.makedirs("test_model/baseline_sensitivity", exist_ok=True)
sensitivity_df.to_csv("test_model/baseline_sensitivity/sensitivity_summary.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
plot_metrics = [
    ("total_km", "Total transport distance (km)"),
    ("total_trips", "Total trips"),
    ("total_co2_tCO2", "Total CO2 emissions (tCO2)"),
    ("total_tonnes", "Total tonnes moved"),
]
for ax, (metric, title) in zip(axes.flatten(), plot_metrics):
    colors = [SENSITIVITY_COLORS[fam] for fam in sensitivity_df["family"]]
    ax.bar(sensitivity_df["label"], sensitivity_df[metric], color=colors, alpha=0.9)
    ax.set_title(title)
    ax.set_xticklabels(sensitivity_df["label"], rotation=45, ha="right")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("test_model/baseline_sensitivity/sensitivity_totals.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
delta_metrics = [
    ("delta_total_km_pct", "Change vs baseline: km"),
    ("delta_total_trips_pct", "Change vs baseline: trips"),
    ("delta_total_co2_tCO2_pct", "Change vs baseline: CO2"),
]
for ax, (metric, title) in zip(axes, delta_metrics):
    subset = sensitivity_df[sensitivity_df["case"] != "baseline_ref"].copy()
    colors = [SENSITIVITY_COLORS[fam] for fam in subset["family"]]
    ax.barh(subset["label"], subset[metric], color=colors, alpha=0.9)
    ax.axvline(0, color="#555555", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Percent change")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("test_model/baseline_sensitivity/sensitivity_deltas.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved baseline sensitivity outputs: test_model/baseline_sensitivity/sensitivity_summary.csv, sensitivity_totals.png, sensitivity_deltas.png")
sensitivity_df


from itertools import product

COMBINATION_LEVELS = {
    "onsite_direct_reuse": [0.05, 0.15, 0.30],
    "ingoing_load_factor": [0.875, 0.90, 0.95],
    "outgoing_load_factor": [0.60, 0.75, 0.90],
    "ev_level": ["current", "improved", "high"],
}

combo_rows = []
for onsite_value, inload_value, outload_value, ev_level in product(
    COMBINATION_LEVELS["onsite_direct_reuse"],
    COMBINATION_LEVELS["ingoing_load_factor"],
    COMBINATION_LEVELS["outgoing_load_factor"],
    COMBINATION_LEVELS["ev_level"],
):
    case_cfg = {
        "case": f"combo_o{int(round(onsite_value*100)):02d}_in{int(round(inload_value*100)):02d}_out{int(round(outload_value*100)):02d}_{ev_level}",
        "label": f"o={onsite_value:.2f} | in={inload_value:.3f} | out={outload_value:.2f} | {ev_level}",
        "family": "Combination sweep",
        "onsite_direct_reuse": onsite_value,
        "ingoing_load_factor": inload_value,
        "outgoing_load_factor": outload_value,
        "ev_level": ev_level,
    }
    _, _, _, totals_case = run_baseline_sensitivity_case(case_cfg)
    combo_rows.append(totals_case)

combination_df = pd.DataFrame(combo_rows)
for metric in ["total_km", "total_trips", "total_co2_tCO2"]:
    combination_df[f"delta_{metric}_pct"] = 100 * (combination_df[metric] - baseline_row[metric]) / baseline_row[metric]

combination_df.to_csv("test_model/baseline_sensitivity/sensitivity_combinations.csv", index=False)

heatmap_specs = [
    ("delta_total_co2_tCO2_pct", "CO2 change vs baseline (%)", "baseline_sensitivity_heatmap_co2.png"),
    ("delta_total_trips_pct", "Trips change vs baseline (%)", "baseline_sensitivity_heatmap_trips.png"),
    ("delta_total_km_pct", "Transport distance change vs baseline (%)", "baseline_sensitivity_heatmap_km.png"),
]

onsite_levels = COMBINATION_LEVELS["onsite_direct_reuse"]
outload_levels = COMBINATION_LEVELS["outgoing_load_factor"]
inload_levels = COMBINATION_LEVELS["ingoing_load_factor"]
ev_levels = COMBINATION_LEVELS["ev_level"]

for metric, title, filename in heatmap_specs:
    fig, axes = plt.subplots(len(inload_levels), len(ev_levels), figsize=(14, 11), sharex=True, sharey=True)
    axes = np.atleast_2d(axes)

    grid_values = combination_df[metric].to_numpy(dtype=float)
    vmax = np.nanmax(np.abs(grid_values)) if len(grid_values) else 1.0
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    for i, inload_value in enumerate(inload_levels):
        for j, ev_level in enumerate(ev_levels):
            ax = axes[i, j]
            subset = combination_df[
                (combination_df["ingoing_load_factor"] == inload_value) &
                (combination_df["ev_level"] == ev_level)
            ].copy()
            pivot = (subset
                     .pivot(index="onsite_direct_reuse", columns="outgoing_load_factor", values=metric)
                     .reindex(index=onsite_levels, columns=outload_levels))
            im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=-vmax, vmax=vmax, aspect="auto")

            for y, onsite_value in enumerate(onsite_levels):
                for x, outload_value in enumerate(outload_levels):
                    value = pivot.loc[onsite_value, outload_value]
                    ax.text(x, y, f"{value:.1f}", ha="center", va="center", fontsize=8, color="black")

            if i == 0:
                ax.set_title(f"EV: {ev_level}")
            if j == 0:
                ax.set_ylabel(f"Ingoing load {inload_value:.3f}\nOnsite reuse")
            ax.set_xticks(range(len(outload_levels)))
            ax.set_xticklabels([f"{v:.2f}" for v in outload_levels], rotation=0)
            ax.set_yticks(range(len(onsite_levels)))
            ax.set_yticklabels([f"{int(v*100)}%" for v in onsite_levels])
            ax.set_xlabel("Outgoing load factor")

    fig.suptitle(title + "\nRows = ingoing load factor, columns = EV level, cells = onsite reuse x outgoing load factor", fontsize=13, y=0.98)
    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.92)
    cbar.set_label("Percent change vs baseline")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f"test_model/baseline_sensitivity/{filename}", dpi=160, bbox_inches="tight")
    plt.show()

print("Saved combination sweep outputs: test_model/baseline_sensitivity/sensitivity_combinations.csv, baseline_sensitivity_heatmap_co2.png, baseline_sensitivity_heatmap_trips.png, baseline_sensitivity_heatmap_km.png")
combination_df.head()


### Combination heatmaps from saved CSV
- This plotting-only section reads `sensitivity_combinations.csv` and builds 9x9 annotated heatmaps.
- Rows combine onsite reuse with ingoing load factor; columns combine outgoing load factor with EV level.


In [ ]:
# =============================================================================
# 17. COMBINATION HEATMAPS FROM CSV
# =============================================================================

from pathlib import Path

combo_path = Path("test_model/baseline_sensitivity/sensitivity_combinations.csv")
if not combo_path.exists():
    raise FileNotFoundError(f"Missing combination sweep CSV: {combo_path}")

combo_plot_df = pd.read_csv(combo_path)

onsite_levels = [0.05, 0.15, 0.30]
inload_levels = [0.875, 0.90, 0.95]
outload_levels = [0.60, 0.75, 0.90]
ev_levels = ["current", "improved", "high"]

row_keys = [f"Onsite {int(o*100)}% | In-load {i:.3f}" for o in onsite_levels for i in inload_levels]
col_keys = [f"Out-load {o:.2f} | EV {e}" for o in outload_levels for e in ev_levels]
row_tick_labels = [f"{int(round(i*100))}%" for o in onsite_levels for i in inload_levels]
col_tick_labels = [e.title() for o in outload_levels for e in ev_levels]

combo_plot_df["row_key"] = combo_plot_df.apply(
    lambda r: f"Onsite {int(r['onsite_direct_reuse']*100)}% | In-load {r['ingoing_load_factor']:.3f}", axis=1
)
combo_plot_df["col_key"] = combo_plot_df.apply(
    lambda r: f"Out-load {r['outgoing_load_factor']:.2f} | EV {r['ev_level']}", axis=1
)

heatmap_configs = [
    ("delta_total_co2_tCO2_pct", "CO2 change vs baseline (%)", "baseline_sensitivity_grid_co2.png"),
    ("delta_total_trips_pct", "Trips change vs baseline (%)", "baseline_sensitivity_grid_trips.png"),
    ("delta_total_km_pct", "Total km change vs baseline (%)", "baseline_sensitivity_grid_km.png"),
]

for metric, title, filename in heatmap_configs:
    pivot = (combo_plot_df
             .pivot(index="row_key", columns="col_key", values=metric)
             .reindex(index=row_keys, columns=col_keys))

    values = pivot.to_numpy(dtype=float)
    vmax = np.nanmax(np.abs(values)) if np.isfinite(values).any() else 1.0
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    fig, ax = plt.subplots(figsize=(18, 12))
    im = ax.imshow(values, cmap="RdYlGn_r", vmin=-vmax, vmax=vmax, aspect="auto")

    ax.set_title(title, fontsize=14, pad=14)
    ax.set_xticks(range(len(col_keys)))
    ax.set_xticklabels(col_tick_labels, rotation=0, ha="center", fontsize=9, fontweight="bold")
    ax.set_yticks(range(len(row_keys)))
    ax.set_yticklabels(row_tick_labels, fontsize=9, fontweight="bold")

    for y in range(values.shape[0]):
        for x in range(values.shape[1]):
            val = values[y, x]
            text = "" if pd.isna(val) else f"{val:.1f}"
            ax.text(x, y, text, ha="center", va="center", fontsize=8, color="black", fontweight="normal")

    # draw grid lines so the 9x9 matrix reads more clearly
    ax.set_xticks(np.arange(-0.5, len(col_keys), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_keys), 1), minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=0.8)
    ax.tick_params(which="minor", bottom=False, left=False)
    for xpos in [2.5, 5.5]:
        ax.axvline(xpos, color="white", linewidth=2.4)
    for ypos in [2.5, 5.5]:
        ax.axhline(ypos, color="white", linewidth=2.4)
    for xpos, label in zip([1, 4, 7], ["Outgoing load factor 60%", "Outgoing load factor 75%", "Outgoing load factor 90%"]):
        ax.text(xpos, len(row_keys) + 0.30, label, ha="center", va="center", fontsize=10, fontweight="bold", clip_on=False)
    for ypos, label in zip([1, 4, 7], ["Onsite reuse 5%", "Onsite reuse 15%", "Onsite reuse 30%"]):
        ax.text(-1.10, ypos, label, ha="center", va="center", fontsize=10, fontweight="bold", rotation=90, clip_on=False)

    cbar = fig.colorbar(im, ax=ax, shrink=0.9)
    cbar.set_label("Percent change vs baseline")

    ax.set_xlabel("Ingoing load factor (%)", labelpad=28, fontweight="normal")
    ax.xaxis.set_label_coords(0.10, -0.035)

    ax.set_ylabel("Electrification level", labelpad=28, fontweight="normal")
    ax.yaxis.set_label_coords(-0.05, 0.095)


    plt.tight_layout()
    plt.savefig(f"test_model/baseline_sensitivity/{filename}", dpi=170, bbox_inches="tight")
    plt.show()

print("Saved CSV-based combination heatmaps: baseline_sensitivity_grid_co2.png, baseline_sensitivity_grid_trips.png, baseline_sensitivity_grid_km.png")
